##### About this notebook:

In [ ]:
#--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
# Author:             Erick Rico Esparza
# Dates:              Mar 24 - Apr 7, 2026
# Description:        This notebook develops the final thesis stage by organizing PM extreme episodes into a seasonal onset-based synoptic analysis and then testing whether those seasonal stories are modulated by ENSO.
# Organization:       Tampere University / Institute of Atmospheric Sciences and Climate Change (ICAyCC-UNAM)
#--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

# Stage 5

## 1. Libraries and setup

### 1.1 Importing libraries

In [258]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import xarray as xr

from io import StringIO
import requests

import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
import matplotlib.patches as mpatches

import cartopy.crs as ccrs
import cartopy.feature as cfeature

from scipy.stats import ttest_ind, chi2_contingency, kruskal, chisquare

### 1.2 Plotting style and display defaults

In [2]:
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)
pd.set_option("display.max_rows", 200)

plt.rcParams["figure.dpi"] = 140
plt.rcParams["savefig.dpi"] = 300
plt.rcParams["axes.titlesize"] = 11
plt.rcParams["axes.labelsize"] = 10
plt.rcParams["xtick.labelsize"] = 9
plt.rcParams["ytick.labelsize"] = 9
plt.rcParams["legend.fontsize"] = 9

print("Section 1 loaded successfully.")

Section 1 loaded successfully.


## 2. Paths, I/O rules, and folder structure

This notebook lives inside `Schedule/Stage 5/`.
All core inputs (CSV + NetCDF files) are located **one level above** (i.e., in `Schedule/`). Some other useful files are located inside `Schedule/Stage 2/outputs/tables`, `Schedule/Stage 3/outputs/tables` and `Schedule/Stage 4/outputs/tables`.

Using `Path` objects and relative paths to keep the project portable.


### 2.1 Working directories, folder structure, and output folders

In [3]:
# Current notebook folder
NOTEBOOK_DIR = Path.cwd().resolve()

# Parent folder containing general inputs and all stage folders
SCHEDULE_DIR = NOTEBOOK_DIR.parent

# Output folders for this final stage
OUTPUT_DIR = NOTEBOOK_DIR / "outputs"
FIGURES_DIR = OUTPUT_DIR / "figures"
TABLES_DIR = OUTPUT_DIR / "tables"
DATA_DIR = OUTPUT_DIR / "data"

for path in [OUTPUT_DIR, FIGURES_DIR, TABLES_DIR, DATA_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print("Notebook directory:", NOTEBOOK_DIR)
print("Schedule directory:", SCHEDULE_DIR)
print("Output directory  :", OUTPUT_DIR)
print("Figures directory :", FIGURES_DIR)
print("Tables directory  :", TABLES_DIR)
print("Data directory    :", DATA_DIR)

Notebook directory: C:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 5
Schedule directory: C:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule
Output directory  : C:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 5\outputs
Figures directory : C:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 5\outputs\figures
Tables directory  : C:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 5\outputs\tables
Data directory    : C:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 5\outputs\data


### 2.2 General input files and inherited stage outputs

In [5]:
# -----------------------------
# Main general input files
# -----------------------------
PM_DAILY_FILE = SCHEDULE_DIR / "pm_cdmx_citymean_daily_2012_2024.csv"

Z500_FILE = SCHEDULE_DIR / "hgt500_mex_2012_2024.nc"
U500_FILE = SCHEDULE_DIR / "uwnd500_mex_2012_2024.nc"
V500_FILE = SCHEDULE_DIR / "vwnd500_mex_2012_2024.nc"

PRECIP_FILE = SCHEDULE_DIR / "precip_sfc_mex_2012_2024.nc"

# -----------------------------
# Prior-stage folders
# -----------------------------
STAGE2_DIR = SCHEDULE_DIR / "Stage 2"
STAGE3_DIR = SCHEDULE_DIR / "Stage 3"
STAGE4_DIR = SCHEDULE_DIR / "Stage 4"

STAGE2_TABLES_DIR = STAGE2_DIR / "outputs" / "tables"
STAGE3_TABLES_DIR = STAGE3_DIR / "outputs" / "tables"
STAGE4_TABLES_DIR = STAGE4_DIR / "outputs" / "tables"

# -----------------------------
# Stage 2 files to reuse
# -----------------------------
STAGE2_PM_DAILY_FLAGS_FILE = STAGE2_TABLES_DIR / "pm_daily_with_p90_flags_2012_2024.csv"

PM10_EPISODES_FILE = STAGE2_TABLES_DIR / "episodes_runs_PM10_p90_with_timefields.csv"
PM25_EPISODES_FILE = STAGE2_TABLES_DIR / "episodes_runs_PM25_p90_with_timefields.csv"

STAGE2_P90_THRESHOLDS_FILE = STAGE2_TABLES_DIR / "p90_thresholds_by_calendar_month_2012_2024.csv"

STAGE2_FREQ_EVENT_DAYS_BY_SEASON_FILE = STAGE2_TABLES_DIR / "freq_event_days_by_season_p90.csv"
STAGE2_SEASONAL_SUMMARY_FILE = STAGE2_TABLES_DIR / "seasonal_summary_p90_eventdays.csv"
STAGE2_SEVERITY_EXCEEDANCE_BY_SEASON_FILE = STAGE2_TABLES_DIR / "severity_exceedance_by_season_p90.csv"

# -----------------------------
# Stage 3 files to reuse
# -----------------------------
STAGE3_PM10_N_CONTROL_FILE = STAGE3_TABLES_DIR / "stage3_N_control_p90_vs_p10_PM10.csv"
STAGE3_PM25_N_CONTROL_FILE = STAGE3_TABLES_DIR / "stage3_N_control_p90_vs_p10_PM25.csv"

STAGE3_FRAC_SIGNIF_BY_MONTH_FILE = STAGE3_TABLES_DIR / "stage3_frac_significant_area_by_month.csv"
STAGE3_SIGFRAC_P90_MINUS_P10_RANKING_FILE = STAGE3_TABLES_DIR / "stage3_sigfrac_p90_minus_p10_ranking.csv"

STAGE3_WHO_BY_MONTH_FILE = STAGE3_TABLES_DIR / "stage3_WHO_exceedance_frequency_magnitude_load_by_month.csv"
STAGE3_PRECIP_CLIM_VOM_FILE = STAGE3_TABLES_DIR / "stage3_precip_climatology_vom_mmday.csv"

# -----------------------------
# Stage 4 files to reuse
# -----------------------------
STAGE4_FINAL_ANALYSIS_FOCUS_FILE = STAGE4_TABLES_DIR / "stage4_final_analysis_focus_table.csv"
STAGE4_SEASONAL_STORY_SUMMARY_FILE = STAGE4_TABLES_DIR / "stage4_seasonal_story_summary.csv"
STAGE4_FOCUS_WINDOW_SAMPLE_SIZES_FILE = STAGE4_TABLES_DIR / "stage4_focus_window_sample_sizes_summary.csv"

STAGE4_CROSS_POLLUTANT_LAG_SUMMARY_FILE = STAGE4_TABLES_DIR / "stage4_cross_pollutant_lag_summary.csv"
STAGE4_LAG_CORE_FINDINGS_FILE = STAGE4_TABLES_DIR / "stage4_lag_core_findings_summary.csv"

STAGE4_TRANSITION_SUMMARY_FILE = STAGE4_TABLES_DIR / "stage4_transition_apr_may_jun_summary.csv"
STAGE4_WETSEASON_JJA_SUMMARY_FILE = STAGE4_TABLES_DIR / "stage4_wetseason_jja_summary.csv"

STAGE4_WORKING_HYPOTHESES_FILE = STAGE4_TABLES_DIR / "stage4_working_hypotheses_table.csv"
STAGE4_STAGE3_MONTHLY_RECAP_FILE = STAGE4_TABLES_DIR / "stage4_stage3_monthly_recap_table.csv"

### 2.3 Reproducibility, seasonal settings, and spatial/domain settings

In [6]:
# Reproducibility
np.random.seed(42)

# Seasons
SEASON_ORDER = ["DJF", "MAM", "JJA", "SON"]

MONTH_TO_SEASON = {
    12: "DJF", 1: "DJF", 2: "DJF",
    3: "MAM", 4: "MAM", 5: "MAM",
    6: "JJA", 7: "JJA", 8: "JJA",
    9: "SON", 10: "SON", 11: "SON",
}

# Main Mexico-centered plotting domain
MEXICO_EXTENT = [-120, -85, 12, 33]   # [lon_min, lon_max, lat_min, lat_max]

# Valley of Mexico reference box
VOM_SW = {"lat": 18.3, "lon": -100.9}
VOM_NE = {"lat": 20.7, "lon": -97.4}

# Optional Valley of Mexico / CDMX reference point
CDMX_STAR = {"lat": 19.4326, "lon": -99.1332}

print("Random seed fixed at 42.")
print("Season order:", SEASON_ORDER)
print("Mexico extent:", MEXICO_EXTENT)
print("Valley of Mexico box:", VOM_SW, "to", VOM_NE)
print("CDMX star:", CDMX_STAR)

Random seed fixed at 42.
Season order: ['DJF', 'MAM', 'JJA', 'SON']
Mexico extent: [-120, -85, 12, 33]
Valley of Mexico box: {'lat': 18.3, 'lon': -100.9} to {'lat': 20.7, 'lon': -97.4}
CDMX star: {'lat': 19.4326, 'lon': -99.1332}


### 2.4 I/O helper functions and reusable utilities

In [7]:
def ensure_datetime_column(df: pd.DataFrame, col: str) -> pd.DataFrame:
    """Return a copy of df with the selected column parsed as datetime."""
    out = df.copy()
    out[col] = pd.to_datetime(out[col])
    return out


def add_time_columns(df: pd.DataFrame, date_col: str) -> pd.DataFrame:
    """Add year, month, and season columns from a datetime column."""
    out = ensure_datetime_column(df, date_col)
    out["year"] = out[date_col].dt.year
    out["month"] = out[date_col].dt.month
    out["season"] = out["month"].map(MONTH_TO_SEASON)
    return out


def month_name_from_number(month: int) -> str:
    """Return English month name from month number."""
    return pd.Timestamp(year=2001, month=month, day=1).strftime("%B")


def save_csv(df: pd.DataFrame, filename: str, index: bool = False) -> Path:
    """Save a DataFrame into outputs/tables and return the saved path."""
    out_path = TABLES_DIR / filename
    df.to_csv(out_path, index=index)
    return out_path


def save_data(obj, filename: str) -> Path:
    """
    Save intermediate data into outputs/data.
    Supports:
    - pandas DataFrame -> .csv
    - xarray Dataset/DataArray -> .nc
    """
    out_path = DATA_DIR / filename

    if isinstance(obj, pd.DataFrame):
        if out_path.suffix != ".csv":
            out_path = out_path.with_suffix(".csv")
        obj.to_csv(out_path, index=False)

    elif isinstance(obj, (xr.Dataset, xr.DataArray)):
        if out_path.suffix != ".nc":
            out_path = out_path.with_suffix(".nc")
        obj.to_netcdf(out_path)

    else:
        raise TypeError("save_data only supports pandas DataFrame or xarray Dataset/DataArray.")

    return out_path


def save_figure(fig: plt.Figure, filename: str, close: bool = True, **kwargs) -> Path:
    """Save a matplotlib figure into outputs/figures and return the saved path."""
    out_path = FIGURES_DIR / filename
    fig.savefig(out_path, bbox_inches="tight", **kwargs)
    if close:
        plt.close(fig)
    return out_path


print(
    "I/O rules:\n"
    "- General inputs are read from the parent Schedule folder.\n"
    "- Prior-stage outputs are read from each stage outputs/tables folder.\n"
    "- New outputs from this notebook are saved into outputs/figures, outputs/tables, and outputs/data.\n"
    "- Important results should be saved to file, not left only as inline notebook output."
)

I/O rules:
- General inputs are read from the parent Schedule folder.
- Prior-stage outputs are read from each stage outputs/tables folder.
- New outputs from this notebook are saved into outputs/figures, outputs/tables, and outputs/data.
- Important results should be saved to file, not left only as inline notebook output.


### 2.5 Existence check for all critical paths

In [8]:
paths = {
    # Main general inputs
    "PM daily": PM_DAILY_FILE,
    "Z500": Z500_FILE,
    "U500": U500_FILE,
    "V500": V500_FILE,
    "Precipitation": PRECIP_FILE,

    # Stage 2
    "Stage2 PM daily flags": STAGE2_PM_DAILY_FLAGS_FILE,
    "Stage2 PM10 episodes": PM10_EPISODES_FILE,
    "Stage2 PM25 episodes": PM25_EPISODES_FILE,
    "Stage2 p90 thresholds": STAGE2_P90_THRESHOLDS_FILE,
    "Stage2 freq by season": STAGE2_FREQ_EVENT_DAYS_BY_SEASON_FILE,
    "Stage2 seasonal summary": STAGE2_SEASONAL_SUMMARY_FILE,
    "Stage2 exceed by season": STAGE2_SEVERITY_EXCEEDANCE_BY_SEASON_FILE,

    # Stage 3
    "Stage3 PM10 N control": STAGE3_PM10_N_CONTROL_FILE,
    "Stage3 PM25 N control": STAGE3_PM25_N_CONTROL_FILE,
    "Stage3 signif by month": STAGE3_FRAC_SIGNIF_BY_MONTH_FILE,
    "Stage3 signif ranking": STAGE3_SIGFRAC_P90_MINUS_P10_RANKING_FILE,
    "Stage3 WHO by month": STAGE3_WHO_BY_MONTH_FILE,
    "Stage3 precip VOM": STAGE3_PRECIP_CLIM_VOM_FILE,

    # Stage 4
    "Stage4 final focus": STAGE4_FINAL_ANALYSIS_FOCUS_FILE,
    "Stage4 seasonal story": STAGE4_SEASONAL_STORY_SUMMARY_FILE,
    "Stage4 focus N summary": STAGE4_FOCUS_WINDOW_SAMPLE_SIZES_FILE,
    "Stage4 lag summary": STAGE4_CROSS_POLLUTANT_LAG_SUMMARY_FILE,
    "Stage4 lag findings": STAGE4_LAG_CORE_FINDINGS_FILE,
    "Stage4 transition summary": STAGE4_TRANSITION_SUMMARY_FILE,
    "Stage4 JJA summary": STAGE4_WETSEASON_JJA_SUMMARY_FILE,
    "Stage4 hypotheses": STAGE4_WORKING_HYPOTHESES_FILE,
    "Stage4 monthly recap": STAGE4_STAGE3_MONTHLY_RECAP_FILE,
}

for name, path in paths.items():
    print(f"{name:<26} -> {path} | exists={path.exists()}")

PM daily                   -> C:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\pm_cdmx_citymean_daily_2012_2024.csv | exists=True
Z500                       -> C:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\hgt500_mex_2012_2024.nc | exists=True
U500                       -> C:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\uwnd500_mex_2012_2024.nc | exists=True
V500                       -> C:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\vwnd500_mex_2012_2024.nc | exists=True
Precipitation              -> C:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\precip_sfc_mex_2012_2024.nc | exists=True
Stage2 PM daily flags      -> C:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 2\outputs\tables\pm_daily_with_p90_flags_2012_2024.csv | exists=True
Stage2 PM10 episodes       -> C:\Users\DELL\OneDrive - T

## 3. Load prior-stage inputs and define final analysis framework

### 3.1 Load daily PM series and episode tables

In [9]:
# Daily PM data
pm_daily = pd.read_csv(PM_DAILY_FILE)
pm_daily.columns = [c.strip() for c in pm_daily.columns]

# Stage 2 event framework
pm_daily_flags = pd.read_csv(STAGE2_PM_DAILY_FLAGS_FILE)
pm_daily_flags.columns = [c.strip() for c in pm_daily_flags.columns]

episodes_pm10 = pd.read_csv(PM10_EPISODES_FILE)
episodes_pm10.columns = [c.strip() for c in episodes_pm10.columns]

episodes_pm25 = pd.read_csv(PM25_EPISODES_FILE)
episodes_pm25.columns = [c.strip() for c in episodes_pm25.columns]

# Parse datetime columns where expected
for df_name, df in [
    ("pm_daily", pm_daily),
    ("pm_daily_flags", pm_daily_flags),
    ("episodes_pm10", episodes_pm10),
    ("episodes_pm25", episodes_pm25),
]:
    for candidate in ["date", "day0", "start", "end"]:
        if candidate in df.columns:
            df[candidate] = pd.to_datetime(df[candidate])

print("Loaded:")
print(f"pm_daily        -> {pm_daily.shape}")
print(f"pm_daily_flags  -> {pm_daily_flags.shape}")
print(f"episodes_pm10   -> {episodes_pm10.shape}")
print(f"episodes_pm25   -> {episodes_pm25.shape}")

Loaded:
pm_daily        -> (4555, 3)
pm_daily_flags  -> (4555, 7)
episodes_pm10   -> (250, 7)
episodes_pm25   -> (286, 7)


### 3.2 Load Stage 3 and Stage 4 summary tables

In [10]:
# Stage 3 summary products
stage3_pm10_n = pd.read_csv(STAGE3_PM10_N_CONTROL_FILE)
stage3_pm25_n = pd.read_csv(STAGE3_PM25_N_CONTROL_FILE)
stage3_signif_by_month = pd.read_csv(STAGE3_FRAC_SIGNIF_BY_MONTH_FILE)
stage3_signif_ranking = pd.read_csv(STAGE3_SIGFRAC_P90_MINUS_P10_RANKING_FILE)
stage3_who_by_month = pd.read_csv(STAGE3_WHO_BY_MONTH_FILE)
stage3_precip_vom = pd.read_csv(STAGE3_PRECIP_CLIM_VOM_FILE)

# Stage 4 support tables
stage4_final_focus = pd.read_csv(STAGE4_FINAL_ANALYSIS_FOCUS_FILE)
stage4_seasonal_story = pd.read_csv(STAGE4_SEASONAL_STORY_SUMMARY_FILE)
stage4_focus_window_n = pd.read_csv(STAGE4_FOCUS_WINDOW_SAMPLE_SIZES_FILE)
stage4_lag_summary = pd.read_csv(STAGE4_CROSS_POLLUTANT_LAG_SUMMARY_FILE)
stage4_lag_findings = pd.read_csv(STAGE4_LAG_CORE_FINDINGS_FILE)
stage4_transition_summary = pd.read_csv(STAGE4_TRANSITION_SUMMARY_FILE)
stage4_jja_summary = pd.read_csv(STAGE4_WETSEASON_JJA_SUMMARY_FILE)
stage4_hypotheses = pd.read_csv(STAGE4_WORKING_HYPOTHESES_FILE)
stage4_monthly_recap = pd.read_csv(STAGE4_STAGE3_MONTHLY_RECAP_FILE)

print("Stage 3 tables loaded:")
print(f"stage3_pm10_n         -> {stage3_pm10_n.shape}")
print(f"stage3_pm25_n         -> {stage3_pm25_n.shape}")
print(f"stage3_signif_by_month-> {stage3_signif_by_month.shape}")
print(f"stage3_signif_ranking -> {stage3_signif_ranking.shape}")
print(f"stage3_who_by_month   -> {stage3_who_by_month.shape}")
print(f"stage3_precip_vom     -> {stage3_precip_vom.shape}")

print("\nStage 4 tables loaded:")
print(f"stage4_final_focus    -> {stage4_final_focus.shape}")
print(f"stage4_seasonal_story -> {stage4_seasonal_story.shape}")
print(f"stage4_focus_window_n -> {stage4_focus_window_n.shape}")
print(f"stage4_lag_summary    -> {stage4_lag_summary.shape}")
print(f"stage4_lag_findings   -> {stage4_lag_findings.shape}")
print(f"stage4_transition     -> {stage4_transition_summary.shape}")
print(f"stage4_jja_summary    -> {stage4_jja_summary.shape}")
print(f"stage4_hypotheses     -> {stage4_hypotheses.shape}")
print(f"stage4_monthly_recap  -> {stage4_monthly_recap.shape}")

Stage 3 tables loaded:
stage3_pm10_n         -> (12, 6)
stage3_pm25_n         -> (12, 6)
stage3_signif_by_month-> (24, 4)
stage3_signif_ranking -> (24, 6)
stage3_who_by_month   -> (12, 8)
stage3_precip_vom     -> (12, 3)

Stage 4 tables loaded:
stage4_final_focus    -> (3, 4)
stage4_seasonal_story -> (4, 10)
stage4_focus_window_n -> (3, 11)
stage4_lag_summary    -> (2, 5)
stage4_lag_findings   -> (2, 5)
stage4_transition     -> (3, 12)
stage4_jja_summary    -> (3, 12)
stage4_hypotheses     -> (3, 4)
stage4_monthly_recap  -> (12, 12)


### 3.3 Inspection of available columns

In [11]:
def show_columns(df: pd.DataFrame, name: str):
    print(f"\n{name} columns:")
    print(list(df.columns))

In [12]:
show_columns(pm_daily, "pm_daily")
show_columns(pm_daily_flags, "pm_daily_flags")
show_columns(episodes_pm10, "episodes_pm10")
show_columns(episodes_pm25, "episodes_pm25")
show_columns(stage3_who_by_month, "stage3_who_by_month")
show_columns(stage4_final_focus, "stage4_final_focus")


pm_daily columns:
['DATE', 'PM10', 'PM2.5']

pm_daily_flags columns:
['DATE', 'PM10', 'PM2.5', 'thr_p90_PM10', 'thr_p90_PM2.5', 'evt_PM10_p90', 'evt_PM2.5_p90']

episodes_pm10 columns:
['episode_id', 'start', 'end', 'duration_days', 'start_year', 'start_month', 'start_season']

episodes_pm25 columns:
['episode_id', 'start', 'end', 'duration_days', 'start_year', 'start_month', 'start_season']

stage3_who_by_month columns:
['month', 'month_name', 'PM10_exceed_pct', 'PM10_excess_mag_mean', 'PM10_excess_load_sum', 'PM25_exceed_pct', 'PM25_excess_mag_mean', 'PM25_excess_load_sum']

stage4_final_focus columns:
['analysis_window', 'stage4_role', 'why_it_stays', 'planned_use_in_stage4']


### 3.4 Build the final event framework from Stage 2 onset-day tables

In [14]:
# In the Stage 2 episode tables, the episode onset is given by "start".
# For the final stage, we explicitly rename that onset date as "day0"
# to keep the lag-composite logic consistent with the thesis narrative.

required_episode_cols = ["episode_id", "start"]
for col in required_episode_cols:
    if col not in episodes_pm10.columns:
        raise ValueError(f"Column '{col}' not found in episodes_pm10.")
    if col not in episodes_pm25.columns:
        raise ValueError(f"Column '{col}' not found in episodes_pm25.")

episodes_pm10_final = episodes_pm10.copy()
episodes_pm25_final = episodes_pm25.copy()

# Ensure datetime
episodes_pm10_final["start"] = pd.to_datetime(episodes_pm10_final["start"])
episodes_pm25_final["start"] = pd.to_datetime(episodes_pm25_final["start"])

# Define onset day explicitly
episodes_pm10_final["day0"] = episodes_pm10_final["start"]
episodes_pm25_final["day0"] = episodes_pm25_final["start"]

# Add time columns from onset day
episodes_pm10_final = add_time_columns(episodes_pm10_final, "day0")
episodes_pm25_final = add_time_columns(episodes_pm25_final, "day0")

# Keep a compact final-stage version
pm10_keep_cols = [c for c in [
    "episode_id", "start", "end", "duration_days",
    "day0", "year", "month", "season",
    "start_year", "start_month", "start_season"
] if c in episodes_pm10_final.columns]

pm25_keep_cols = [c for c in [
    "episode_id", "start", "end", "duration_days",
    "day0", "year", "month", "season",
    "start_year", "start_month", "start_season"
] if c in episodes_pm25_final.columns]

pm10_onset_framework = episodes_pm10_final[pm10_keep_cols].copy()
pm25_onset_framework = episodes_pm25_final[pm25_keep_cols].copy()

print("Final onset framework created.")
print(f"PM10 onset framework -> {pm10_onset_framework.shape}")
print(f"PM25 onset framework -> {pm25_onset_framework.shape}")

pm10_onset_framework.head()

Final onset framework created.
PM10 onset framework -> (250, 11)
PM25 onset framework -> (286, 11)


,episode_id,start,end,duration_days,day0,year,month,season,start_year,start_month,start_season
0,1,2012-01-01,2012-01-01,1,2012-01-01,2012,1,DJF,2012,1,DJF
1,2,2012-01-20,2012-01-21,2,2012-01-20,2012,1,DJF,2012,1,DJF
2,3,2012-01-23,2012-01-24,2,2012-01-23,2012,1,DJF,2012,1,DJF
3,4,2012-03-01,2012-03-03,3,2012-03-01,2012,3,MAM,2012,3,MAM
4,5,2012-03-06,2012-03-09,4,2012-03-06,2012,3,MAM,2012,3,MAM


### 3.5 Define seasons, analysis windows, and seasonal subsets

In [17]:
# SON is included in the full framework for completeness,
# even if later it may play a more secondary narrative role.

ANALYSIS_SEASONS = ["DJF", "MAM", "JJA", "SON"]
CORE_NARRATIVE_SEASONS = ["DJF", "MAM", "JJA"]

# Seasonal onset subsets
pm10_onset_by_season = {
    season: pm10_onset_framework.loc[pm10_onset_framework["season"] == season].copy()
    for season in ANALYSIS_SEASONS
}

pm25_onset_by_season = {
    season: pm25_onset_framework.loc[pm25_onset_framework["season"] == season].copy()
    for season in ANALYSIS_SEASONS
}

# Seasonal windows
FOCUS_MONTHS = {
    "DJF_benchmark": [12, 1, 2],
    "MAM_transition": [3, 4, 5],
    "JJA_wet_contrast": [6, 7, 8],
    "SON_context": [9, 10, 11],
}

# Anchor months inherited from Stage 4
ANCHOR_MONTHS = {
    "February": 2,
    "May": 5,
    "July": 7,
}

# Seasonal onset counts
pm10_n_by_season = (
    pm10_onset_framework.groupby("season")
    .size()
    .reindex(ANALYSIS_SEASONS)
    .reset_index(name="PM10_onset_days")
)

pm25_n_by_season = (
    pm25_onset_framework.groupby("season")
    .size()
    .reindex(ANALYSIS_SEASONS)
    .reset_index(name="PM25_onset_days")
)

seasonal_onset_n = pm10_n_by_season.merge(pm25_n_by_season, on="season", how="outer")

# Saving framework table
save_csv(seasonal_onset_n, "final_stage_onset_counts_by_season.csv", index=False)

# Showing framework table
print("Seasonal subsets defined.")
seasonal_onset_n

Seasonal subsets defined.


,season,PM10_onset_days,PM25_onset_days
0,DJF,70,74
1,JJA,68,76
2,MAM,53,63
3,SON,59,73


## 4. Load reanalysis fields and preprocess analysis-ready datasets

### 4.1 Load 500 hPa fields and precipitation

In [18]:
z500_ds = xr.open_dataset(Z500_FILE)
u500_ds = xr.open_dataset(U500_FILE)
v500_ds = xr.open_dataset(V500_FILE)
precip_ds = xr.open_dataset(PRECIP_FILE)

print("Loaded datasets:")
print("z500_ds  :", z500_ds.dims)
print("u500_ds  :", u500_ds.dims)
print("v500_ds  :", v500_ds.dims)
print("precip_ds:", precip_ds.dims)

Loaded datasets:
z500_ds  : FrozenMappingWarningOnValuesAccess({'time': 4749, 'y': 100, 'x': 145})
u500_ds  : FrozenMappingWarningOnValuesAccess({'time': 4749, 'y': 100, 'x': 145})
v500_ds  : FrozenMappingWarningOnValuesAccess({'time': 4749, 'y': 100, 'x': 145})
precip_ds: FrozenMappingWarningOnValuesAccess({'time': 4749, 'y': 90, 'x': 140})


### 4.2 Inspection of coordinate structure

In [21]:
print("Z500 coords:", list(z500_ds.coords))
print("U500 coords:", list(u500_ds.coords))
print("V500 coords:", list(v500_ds.coords))
print("Precip coords:", list(precip_ds.coords))

print("\nZ500 dims:", z500_ds.dims)
print("U500 dims:", u500_ds.dims)
print("V500 dims:", v500_ds.dims)
print("Precip dims:", precip_ds.dims)

print("\nZ500 variable names:", list(z500_ds.data_vars))
print("Precip variable names:", list(precip_ds.data_vars))

Z500 coords: ['time', 'level', 'y', 'x', 'lat', 'lon']
U500 coords: ['time', 'level', 'y', 'x', 'lat', 'lon']
V500 coords: ['time', 'level', 'y', 'x', 'lat', 'lon']
Precip coords: ['time', 'lat', 'lon', 'y', 'x']

Z500 dims: FrozenMappingWarningOnValuesAccess({'time': 4749, 'y': 100, 'x': 145})
U500 dims: FrozenMappingWarningOnValuesAccess({'time': 4749, 'y': 100, 'x': 145})
V500 dims: FrozenMappingWarningOnValuesAccess({'time': 4749, 'y': 100, 'x': 145})
Precip dims: FrozenMappingWarningOnValuesAccess({'time': 4749, 'y': 90, 'x': 140})

Z500 variable names: ['hgt']
Precip variable names: ['precip_mmday']


### 4.3 Standardize coordinates and subset the analysis domain

In [22]:
def wrap_longitudes_to_180(ds: xr.Dataset) -> xr.Dataset:
    """
    Convert longitudes from 0–360 to -180–180 if needed.
    Works for both 1D and 2D longitude coordinates.
    """
    lon = ds["lon"]

    if float(lon.max()) > 180:
        ds = ds.assign_coords(lon=((lon + 180) % 360) - 180)

    return ds


def subset_domain_2d(ds: xr.Dataset, extent: list) -> xr.Dataset:
    """
    Subset a dataset with 2D lon/lat coordinates using a spatial mask.
    Keeps the original x/y grid and drops points outside the domain.
    """
    lon_min, lon_max, lat_min, lat_max = extent

    lon2d = ds["lon"]
    lat2d = ds["lat"]

    mask = (
        (lon2d >= lon_min) & (lon2d <= lon_max) &
        (lat2d >= lat_min) & (lat2d <= lat_max)
    )

    return ds.where(mask, drop=True)

In [23]:
# Convert longitude convention if needed
z500_ds = wrap_longitudes_to_180(z500_ds)
u500_ds = wrap_longitudes_to_180(u500_ds)
v500_ds = wrap_longitudes_to_180(v500_ds)
precip_ds = wrap_longitudes_to_180(precip_ds)

# Subset to the Mexico-centered analysis domain
z500_ds = subset_domain_2d(z500_ds, MEXICO_EXTENT)
u500_ds = subset_domain_2d(u500_ds, MEXICO_EXTENT)
v500_ds = subset_domain_2d(v500_ds, MEXICO_EXTENT)
precip_ds = subset_domain_2d(precip_ds, MEXICO_EXTENT)

print("Subsetted datasets:")
print("z500_ds  :", z500_ds.dims)
print("u500_ds  :", u500_ds.dims)
print("v500_ds  :", v500_ds.dims)
print("precip_ds:", precip_ds.dims)

Subsetted datasets:
z500_ds  : FrozenMappingWarningOnValuesAccess({'time': 4749, 'y': 90, 'x': 140})
u500_ds  : FrozenMappingWarningOnValuesAccess({'time': 4749, 'y': 90, 'x': 140})
v500_ds  : FrozenMappingWarningOnValuesAccess({'time': 4749, 'y': 90, 'x': 140})
precip_ds: FrozenMappingWarningOnValuesAccess({'time': 4749, 'y': 90, 'x': 140})


### 4.4 Time alignment

In [24]:
# Standardize PM date columns before merges/alignment
pm_daily = pm_daily.rename(columns={"DATE": "date"})
pm_daily_flags = pm_daily_flags.rename(columns={"DATE": "date"})

pm_daily["date"] = pd.to_datetime(pm_daily["date"])
pm_daily_flags["date"] = pd.to_datetime(pm_daily_flags["date"])

# Find common time coverage across PM and meteorology
common_time = (
    pd.Index(pm_daily["date"])
    .intersection(pd.Index(z500_ds["time"].values))
    .intersection(pd.Index(u500_ds["time"].values))
    .intersection(pd.Index(v500_ds["time"].values))
    .intersection(pd.Index(precip_ds["time"].values))
)

common_time = pd.DatetimeIndex(sorted(common_time))

# Restrict all tables/datasets to common dates
pm_daily = pm_daily.loc[pm_daily["date"].isin(common_time)].copy()
pm_daily_flags = pm_daily_flags.loc[pm_daily_flags["date"].isin(common_time)].copy()

z500_ds = z500_ds.sel(time=common_time)
u500_ds = u500_ds.sel(time=common_time)
v500_ds = v500_ds.sel(time=common_time)
precip_ds = precip_ds.sel(time=common_time)

print("Common time coverage:")
print("start :", common_time.min())
print("end   :", common_time.max())
print("n days:", len(common_time))

print("\nPM missing values after common-time filtering:")
print(pm_daily[["PM10", "PM2.5"]].isna().sum())

print("\nTime sizes after filtering:")
print("z500  :", z500_ds.sizes["time"])
print("u500  :", u500_ds.sizes["time"])
print("v500  :", v500_ds.sizes["time"])
print("precip:", precip_ds.sizes["time"])

Common time coverage:
start : 2012-01-01 00:00:00
end   : 2024-12-31 00:00:00
n days: 4555

PM missing values after common-time filtering:
PM10     0
PM2.5    0
dtype: int64

Time sizes after filtering:
z500  : 4555
u500  : 4555
v500  : 4555
precip: 4555


### 4.5 Valley of Mexico mask and regional precipitation series

In [25]:
def subset_vom_box_2d(ds: xr.Dataset, sw: dict, ne: dict) -> xr.Dataset:
    """Subset dataset to the Valley of Mexico reference box using a 2D mask."""
    lon2d = ds["lon"]
    lat2d = ds["lat"]

    mask = (
        (lon2d >= sw["lon"]) & (lon2d <= ne["lon"]) &
        (lat2d >= sw["lat"]) & (lat2d <= ne["lat"])
    )

    return ds.where(mask, drop=True)

In [26]:
precip_vom_ds = subset_vom_box_2d(precip_ds, VOM_SW, VOM_NE)

PRECIP_VAR = "precip_mmday"
print("Using precipitation variable:", PRECIP_VAR)

precip_vom_daily = (
    precip_vom_ds[PRECIP_VAR]
    .mean(dim=("y", "x"))
    .to_dataframe(name="precip_vom_mean")
    .reset_index()
)

precip_vom_daily["time"] = pd.to_datetime(precip_vom_daily["time"])
precip_vom_daily["month"] = precip_vom_daily["time"].dt.month
precip_vom_daily["season"] = precip_vom_daily["month"].map(MONTH_TO_SEASON)

save_csv(precip_vom_daily, "precip_vom_daily_common_period.csv", index=False)

print("Saved: precip_vom_daily_common_period.csv")
precip_vom_daily.head()

Using precipitation variable: precip_mmday
Saved: precip_vom_daily_common_period.csv


,time,precip_vom_mean,month,season
0,2012-01-01,0.394031,1,DJF
1,2012-01-02,0.537714,1,DJF
2,2012-01-03,-0.002538,1,DJF
3,2012-01-04,0.045473,1,DJF
4,2012-01-05,-0.027926,1,DJF


In [27]:
list(u500_ds.data_vars), list(v500_ds.data_vars)

(['uwnd'], ['vwnd'])

### 4.6 Compute monthly climatologies and anomaly fields

In [28]:
# Variable names confirmed from the datasets
Z500_VAR = "hgt"
U500_VAR = "uwnd"
V500_VAR = "vwnd"
PRECIP_VAR = "precip_mmday"

print("Using variables:")
print("Z500  :", Z500_VAR)
print("U500  :", U500_VAR)
print("V500  :", V500_VAR)
print("Precip:", PRECIP_VAR)

# -----------------------------
# Clean precipitation series
# -----------------------------
# Small negative daily values can appear as numerical artifacts.
# They are not physically meaningful for precipitation, so clip them to zero.

precip_ds[PRECIP_VAR] = precip_ds[PRECIP_VAR].clip(min=0)
precip_vom_daily["precip_vom_mean"] = precip_vom_daily["precip_vom_mean"].clip(lower=0)

# -----------------------------
# Monthly climatologies
# -----------------------------
z500_clim_monthly = z500_ds[Z500_VAR].groupby("time.month").mean("time")
u500_clim_monthly = u500_ds[U500_VAR].groupby("time.month").mean("time")
v500_clim_monthly = v500_ds[V500_VAR].groupby("time.month").mean("time")
precip_clim_monthly = precip_ds[PRECIP_VAR].groupby("time.month").mean("time")

# -----------------------------
# Monthly anomalies
# -----------------------------
z500_anom = z500_ds[Z500_VAR].groupby("time.month") - z500_clim_monthly
u500_anom = u500_ds[U500_VAR].groupby("time.month") - u500_clim_monthly
v500_anom = v500_ds[V500_VAR].groupby("time.month") - v500_clim_monthly

# -----------------------------
# Valley of Mexico monthly precipitation climatology
# -----------------------------
precip_vom_clim_monthly = (
    precip_vom_daily.groupby("month", as_index=False)["precip_vom_mean"]
    .mean()
    .rename(columns={"precip_vom_mean": "precip_vom_mean_mmday"})
)

precip_vom_clim_monthly["month_name"] = (
    precip_vom_clim_monthly["month"].apply(month_name_from_number)
)

# Reorder columns
precip_vom_clim_monthly = precip_vom_clim_monthly[
    ["month", "month_name", "precip_vom_mean_mmday"]
]

# -----------------------------
# Save key outputs
# -----------------------------
save_data(z500_anom.to_dataset(name="z500_anom"), "z500_monthly_anomalies.nc")
save_data(u500_anom.to_dataset(name="u500_anom"), "u500_monthly_anomalies.nc")
save_data(v500_anom.to_dataset(name="v500_anom"), "v500_monthly_anomalies.nc")

save_data(z500_clim_monthly.to_dataset(name="z500_clim_monthly"), "z500_monthly_climatology.nc")
save_data(u500_clim_monthly.to_dataset(name="u500_clim_monthly"), "u500_monthly_climatology.nc")
save_data(v500_clim_monthly.to_dataset(name="v500_clim_monthly"), "v500_monthly_climatology.nc")
save_data(precip_clim_monthly.to_dataset(name="precip_clim_monthly"), "precip_monthly_climatology.nc")

save_csv(precip_vom_daily, "precip_vom_daily_common_period.csv", index=False)
save_csv(precip_vom_clim_monthly, "precip_vom_monthly_climatology_common_period.csv", index=False)

# -----------------------------
# Quick checks
# -----------------------------
print("Saved anomaly fields and monthly climatology files.")
print("\nShapes:")
print("z500_clim_monthly :", z500_clim_monthly.shape)
print("u500_clim_monthly :", u500_clim_monthly.shape)
print("v500_clim_monthly :", v500_clim_monthly.shape)
print("precip_clim_monthly:", precip_clim_monthly.shape)
print("z500_anom         :", z500_anom.shape)
print("u500_anom         :", u500_anom.shape)
print("v500_anom         :", v500_anom.shape)

precip_vom_clim_monthly

Using variables:
Z500  : hgt
U500  : uwnd
V500  : vwnd
Precip: precip_mmday
Saved anomaly fields and monthly climatology files.

Shapes:
z500_clim_monthly : (12, 90, 140)
u500_clim_monthly : (12, 90, 140)
v500_clim_monthly : (12, 90, 140)
precip_clim_monthly: (12, 90, 140)
z500_anom         : (4555, 90, 140)
u500_anom         : (4555, 90, 140)
v500_anom         : (4555, 90, 140)


,month,month_name,precip_vom_mean_mmday
0,1,January,0.314199
1,2,February,0.333571
2,3,March,0.708063
3,4,April,0.810095
4,5,May,2.015312
5,6,June,4.442248
6,7,July,4.648746
7,8,August,4.951902
8,9,September,5.228750
9,10,October,2.475214


### 4.7 Check for meteorology and derived fields

In [29]:
def qc_summary_da(da: xr.DataArray, name: str) -> pd.DataFrame:
    """Return a compact QC summary for an xarray DataArray."""
    valid_fraction = float(da.notnull().mean().values)
    data_min = float(da.min(skipna=True).values)
    data_max = float(da.max(skipna=True).values)

    return pd.DataFrame({
        "field": [name],
        "valid_fraction": [valid_fraction],
        "min": [data_min],
        "max": [data_max],
    })

In [30]:
qc_table = pd.concat([
    qc_summary_da(z500_ds[Z500_VAR], "z500_raw_common"),
    qc_summary_da(u500_ds[U500_VAR], "u500_raw_common"),
    qc_summary_da(v500_ds[V500_VAR], "v500_raw_common"),
    qc_summary_da(precip_ds[PRECIP_VAR], "precip_raw_common_clipped"),
    qc_summary_da(z500_anom, "z500_anom"),
    qc_summary_da(u500_anom, "u500_anom"),
    qc_summary_da(v500_anom, "v500_anom"),
    qc_summary_da(z500_clim_monthly, "z500_clim_monthly"),
    qc_summary_da(u500_clim_monthly, "u500_clim_monthly"),
    qc_summary_da(v500_clim_monthly, "v500_clim_monthly"),
    qc_summary_da(precip_clim_monthly, "precip_clim_monthly"),
], ignore_index=True)

save_csv(qc_table, "meteorology_qc_summary.csv", index=False)

print("Saved: meteorology_qc_summary.csv")
qc_table

Saved: meteorology_qc_summary.csv


,field,valid_fraction,min,max
0,z500_raw_common,0.786032,5367.404785,6007.630859
1,u500_raw_common,0.786032,-30.429220,54.217735
2,v500_raw_common,0.786032,-44.263676,45.515244
3,precip_raw_common_clipped,0.786032,0.000000,331.510681
4,z500_anom,0.786032,-353.995605,202.149414
5,u500_anom,0.786032,-47.802681,39.791740
6,v500_anom,0.786032,-43.800117,43.018269
7,z500_clim_monthly,0.786032,5691.992676,5940.316406
8,u500_clim_monthly,0.786032,-8.652335,24.602713
9,v500_clim_monthly,0.786032,-4.402309,5.812618


In [31]:
# ------------------------------------------------------------
# Precipitation clipping sanity check
# ------------------------------------------------------------

precip_negative_count_after = int((precip_ds[PRECIP_VAR] < 0).sum().values)
precip_vom_negative_count_after = int((precip_vom_daily["precip_vom_mean"] < 0).sum())

print("Negative gridpoint precipitation values after clipping:", precip_negative_count_after)
print("Negative VOM daily mean values after clipping:", precip_vom_negative_count_after)

Negative gridpoint precipitation values after clipping: 0
Negative VOM daily mean values after clipping: 0


## 5. Seasonal descriptive framework before onset analysis

### 5.1 Seasonal sample-size and event coverage summary

In [32]:
# Total available PM days by season from the common-time PM table
pm_daily_with_season = pm_daily.copy()
pm_daily_with_season["month"] = pm_daily_with_season["date"].dt.month
pm_daily_with_season["season"] = pm_daily_with_season["month"].map(MONTH_TO_SEASON)

available_days_by_season = (
    pm_daily_with_season.groupby("season")
    .size()
    .reindex(SEASON_ORDER)
    .reset_index(name="n_available_days")
)

# Onset counts by season already built in Section 3
seasonal_framework = available_days_by_season.merge(
    seasonal_onset_n.rename(columns={"season": "season"}),
    on="season",
    how="left"
)

# Event coverage as % of available days
seasonal_framework["PM10_onset_pct_of_days"] = (
    100 * seasonal_framework["PM10_onset_days"] / seasonal_framework["n_available_days"]
)

seasonal_framework["PM25_onset_pct_of_days"] = (
    100 * seasonal_framework["PM25_onset_days"] / seasonal_framework["n_available_days"]
)

save_csv(seasonal_framework, "seasonal_event_framework_summary.csv", index=False)

print("Saved: seasonal_event_framework_summary.csv")
seasonal_framework

Saved: seasonal_event_framework_summary.csv


,season,n_available_days,PM10_onset_days,PM25_onset_days,PM10_onset_pct_of_days,PM25_onset_pct_of_days
0,DJF,1126,70,74,6.216696,6.571936
1,MAM,1154,53,63,4.592721,5.459272
2,JJA,1154,68,76,5.892548,6.585789
3,SON,1121,59,73,5.263158,6.512043


### 5.2 Seasonal PM burden and exceedance context

In [33]:
# Add season to Stage 3 monthly WHO table
stage3_who_by_month["season"] = stage3_who_by_month["month"].map(MONTH_TO_SEASON)

# Seasonal mean burden/exceedance context from monthly summaries
seasonal_who_context = (
    stage3_who_by_month
    .groupby("season", as_index=False)[[
        "PM10_exceed_pct",
        "PM10_excess_mag_mean",
        "PM10_excess_load_sum",
        "PM25_exceed_pct",
        "PM25_excess_mag_mean",
        "PM25_excess_load_sum"
    ]]
    .mean()
)

# Keep season order
seasonal_who_context["season"] = pd.Categorical(
    seasonal_who_context["season"],
    categories=SEASON_ORDER,
    ordered=True
)
seasonal_who_context = seasonal_who_context.sort_values("season").reset_index(drop=True)

save_csv(seasonal_who_context, "seasonal_who_context_from_stage3.csv", index=False)

print("Saved: seasonal_who_context_from_stage3.csv")
seasonal_who_context

Saved: seasonal_who_context_from_stage3.csv


,season,PM10_exceed_pct,PM10_excess_mag_mean,PM10_excess_load_sum,PM25_exceed_pct,PM25_excess_mag_mean,PM25_excess_load_sum
0,DJF,73.855756,18.915201,5243.406667,89.528705,13.970249,4723.010000
1,MAM,66.547508,14.787683,3729.316667,95.050711,12.363148,4496.876667
2,JJA,14.789858,6.365254,346.936667,71.526699,6.370492,1721.310000
3,SON,28.132614,10.010783,1248.000000,71.006429,8.471798,2293.496667


### 5.3 Seasonal precipitation context

In [34]:
# Seasonal precipitation climatology over the Valley of Mexico
precip_vom_daily["season"] = precip_vom_daily["month"].map(MONTH_TO_SEASON)

seasonal_precip_context = (
    precip_vom_daily
    .groupby("season", as_index=False)["precip_vom_mean"]
    .mean()
    .rename(columns={"precip_vom_mean": "precip_vom_mean_mmday"})
)

seasonal_precip_context["season"] = pd.Categorical(
    seasonal_precip_context["season"],
    categories=SEASON_ORDER,
    ordered=True
)
seasonal_precip_context = seasonal_precip_context.sort_values("season").reset_index(drop=True)

save_csv(seasonal_precip_context, "seasonal_precip_context_vom.csv", index=False)

print("Saved: seasonal_precip_context_vom.csv")
seasonal_precip_context

Saved: seasonal_precip_context_vom.csv


,season,precip_vom_mean_mmday
0,DJF,0.354234
1,MAM,1.163150
2,JJA,4.688356
3,SON,2.887631


### 5.4 Seasonal narrative anchor table

In [35]:
seasonal_anchor_table = seasonal_framework.merge(
    seasonal_who_context,
    on="season",
    how="left"
).merge(
    seasonal_precip_context,
    on="season",
    how="left"
)

# Add a compact interpretation anchor
anchor_map = {
    "DJF": "Dry benchmark season",
    "MAM": "Transition season",
    "JJA": "Wet-season contrast",
    "SON": "Included for completeness"
}

seasonal_anchor_table["interpretation_anchor"] = seasonal_anchor_table["season"].map(anchor_map)

save_csv(seasonal_anchor_table, "seasonal_narrative_anchor_table.csv", index=False)

print("Saved: seasonal_narrative_anchor_table.csv")
seasonal_anchor_table

Saved: seasonal_narrative_anchor_table.csv


,season,n_available_days,PM10_onset_days,PM25_onset_days,PM10_onset_pct_of_days,PM25_onset_pct_of_days,PM10_exceed_pct,PM10_excess_mag_mean,PM10_excess_load_sum,PM25_exceed_pct,PM25_excess_mag_mean,PM25_excess_load_sum,precip_vom_mean_mmday,interpretation_anchor
0,DJF,1126,70,74,6.216696,6.571936,73.855756,18.915201,5243.406667,89.528705,13.970249,4723.010000,0.354234,Dry benchmark season
1,MAM,1154,53,63,4.592721,5.459272,66.547508,14.787683,3729.316667,95.050711,12.363148,4496.876667,1.163150,Transition season
2,JJA,1154,68,76,5.892548,6.585789,14.789858,6.365254,346.936667,71.526699,6.370492,1721.310000,4.688356,Wet-season contrast
3,SON,1121,59,73,5.263158,6.512043,28.132614,10.010783,1248.000000,71.006429,8.471798,2293.496667,2.887631,Included for completeness


## 6. Seasonal onset-day composites at 500 hPa

### 6.1 Build seasonal onset-day subsets and plotting helpers

In [41]:
# Onset dates by season
pm10_onset_dates_by_season = {
    season: pd.DatetimeIndex(pm10_onset_by_season[season]["day0"].sort_values().unique())
    for season in SEASON_ORDER
}

pm25_onset_dates_by_season = {
    season: pd.DatetimeIndex(pm25_onset_by_season[season]["day0"].sort_values().unique())
    for season in SEASON_ORDER
}

# 2D lon/lat arrays for plotting
lon2d = z500_ds["lon"].values
lat2d = z500_ds["lat"].values

# Reference plotting domain and Valley of Mexico box
DOMAIN_MAIN = {
    "lon_min": MEXICO_EXTENT[0],
    "lon_max": MEXICO_EXTENT[1],
    "lat_min": MEXICO_EXTENT[2],
    "lat_max": MEXICO_EXTENT[3],
}

# Rectangle definition as (x0, y0, width, height)
VOM_BOX = (
    VOM_SW["lon"],
    VOM_SW["lat"],
    VOM_NE["lon"] - VOM_SW["lon"],
    VOM_NE["lat"] - VOM_SW["lat"],
)

CDMX_REF = {
    "lon": CDMX_STAR["lon"],
    "lat": CDMX_STAR["lat"],
}

SEASON_ROLE_MAP = {
    "DJF": "dry benchmark",
    "MAM": "transition season",
    "JJA": "wet-season contrast",
    "SON": "included for completeness",
}

In [ ]:
def composite_from_dates(da: xr.DataArray, dates: pd.DatetimeIndex) -> xr.DataArray:
    """Compute mean composite over selected dates."""
    return da.sel(time=dates).mean("time")

def get_seasonal_composite_dict(z_anom, u_anom, v_anom, onset_dates_by_season):
    """
    Build a seasonal composite dictionary with:
    - dates
    - n
    - z, u, v composites
    """
    out = {}
    for season in SEASON_ORDER:
        dates = onset_dates_by_season[season]
        out[season] = {
            "dates": dates,
            "n": len(dates),
            "z": composite_from_dates(z_anom, dates),
            "u": composite_from_dates(u_anom, dates),
            "v": composite_from_dates(v_anom, dates),
        }
    return out

def build_seasonal_summary_table(comp_dict, pollutant_label):
    """
    Build a compact table with:
    - N
    - max abs anomaly
    - domain-mean abs anomaly
    - domain-mean wind anomaly magnitude
    """
    rows = []

    for season in SEASON_ORDER:
        z_vals = comp_dict[season]["z"].values
        u_vals = comp_dict[season]["u"].values
        v_vals = comp_dict[season]["v"].values

        max_abs_z = float(np.nanmax(np.abs(z_vals)))
        mean_abs_z = float(np.nanmean(np.abs(z_vals)))
        mean_wind_mag = float(np.nanmean(np.sqrt(u_vals**2 + v_vals**2)))

        rows.append({
            "pollutant": pollutant_label,
            "season": season,
            "n_onset_days": comp_dict[season]["n"],
            "max_abs_z500_anom": max_abs_z,
            "mean_abs_z500_anom": mean_abs_z,
            "mean_wind_anom_mag": mean_wind_mag,
        })

    return pd.DataFrame(rows)

def plot_seasonal_composite_panel(
    comp_dict,
    pollutant_label,
    out_fig_path,
    vector_step=6,
    contour_step=5
):
    """
    Export a 4-panel seasonal day-0 composite figure (2x2),
    visually aligned with Stage 4 lag composite figures.

    Shading: Z500 anomaly
    Contours: Z500 anomaly (positive solid / negative dashed)
    Vectors: U500/V500 anomalies
    """

    # Symmetric color range from all seasonal composites
    max_abs = max(
        float(np.nanmax(np.abs(comp_dict[season]["z"].values)))
        for season in SEASON_ORDER
    )
    if not np.isfinite(max_abs) or max_abs == 0:
        max_abs = 1.0

    norm = TwoSlopeNorm(vcenter=0, vmin=-max_abs, vmax=max_abs)

    proj = ccrs.PlateCarree()
    fig, axes = plt.subplots(
        2, 2,
        figsize=(13.5, 10.5),
        dpi=250,
        subplot_kw={"projection": proj}
    )
    axes = axes.flatten()

    # Tighter but still balanced layout
    fig.subplots_adjust(
        left=0.07,
        right=0.93,
        bottom=0.16,
        top=0.81,
        wspace=0.10,
        hspace=0.20
    )

    pcm_ref = None

    for i, season in enumerate(SEASON_ORDER):
        ax = axes[i]

        ax.set_extent(
            [
                DOMAIN_MAIN["lon_min"], DOMAIN_MAIN["lon_max"],
                DOMAIN_MAIN["lat_min"], DOMAIN_MAIN["lat_max"]
            ],
            crs=proj
        )

        ax.coastlines(resolution="50m", linewidth=0.5)
        ax.add_feature(cfeature.BORDERS, linewidth=0.4)
        ax.add_feature(cfeature.STATES.with_scale("50m"), linewidth=0.3)

        z = comp_dict[season]["z"].values
        u = comp_dict[season]["u"].values
        v = comp_dict[season]["v"].values
        n_used = comp_dict[season]["n"]

        # --- Shading
        pcm = ax.pcolormesh(
            lon2d, lat2d, z,
            cmap="RdBu_r",
            norm=norm,
            shading="auto",
            transform=proj
        )
        pcm_ref = pcm

        # --- Contours (positive solid / negative dashed)
        lev = np.arange(-max_abs, max_abs + contour_step, contour_step)
        pos = lev[lev > 0]
        neg = lev[lev < 0]

        if len(pos) > 0:
            ax.contour(
                lon2d, lat2d, z,
                levels=pos,
                colors="k",
                linewidths=0.7,
                transform=proj
            )

        if len(neg) > 0:
            ax.contour(
                lon2d, lat2d, z,
                levels=neg,
                colors="k",
                linewidths=0.7,
                linestyles="--",
                transform=proj
            )

        # --- Vectors
        yy = np.arange(0, z.shape[0], vector_step)
        xx = np.arange(0, z.shape[1], vector_step)

        ax.quiver(
            lon2d[np.ix_(yy, xx)],
            lat2d[np.ix_(yy, xx)],
            u[np.ix_(yy, xx)],
            v[np.ix_(yy, xx)],
            transform=proj,
            color="black",
            scale=80,
            width=0.0022,
            headwidth=3.5,
            headlength=4.5,
            headaxislength=4.0,
            minlength=0.05,
            pivot="middle",
            zorder=4
        )

        # --- Valley of Mexico box
        rect = mpatches.Rectangle(
            (VOM_BOX[0], VOM_BOX[1]),
            VOM_BOX[2], VOM_BOX[3],
            fill=False,
            edgecolor="k",
            linewidth=1.0,
            linestyle="-",
            transform=proj
        )
        ax.add_patch(rect)

        # --- CDMX star
        ax.plot(
            CDMX_REF["lon"], CDMX_REF["lat"],
            marker="*",
            color="gold",
            markersize=9,
            markeredgecolor="k",
            transform=proj
        )

        # --- Panel title
        ax.set_title(
            f"{season} (N={n_used})",
            fontsize=14,
            weight="bold"
        )

        # --- Gridlines with lat/lon labels on all subplots
        gl = ax.gridlines(
            draw_labels=True,
            linewidth=0.2,
            color="gray",
            alpha=0.5,
            linestyle="--"
        )
        gl.top_labels = False
        gl.right_labels = False
        gl.left_labels = True
        gl.bottom_labels = True

    # --- Bottom colorbar
    cbar_ax = fig.add_axes([0.30, 0.07, 0.40, 0.028])
    cb = fig.colorbar(pcm_ref, cax=cbar_ax, orientation="horizontal", extend="both")
    cb.set_label("Z500′ anomaly (m)", fontsize=12)
    cb.ax.tick_params(labelsize=10)

    # --- Titles
    total_n = sum(comp_dict[s]["n"] for s in SEASON_ORDER)

    fig.suptitle(
        f"{pollutant_label}: seasonal day-0 composites at 500 hPa (2012–2024)",
        fontsize=20,
        weight="bold",
        x=0.5,
        y=0.975
    )

    fig.text(
        0.5, 0.928,
        "Shading/contours: Z500′ anomaly | Vectors: Δ(U′,V′) at 500 hPa | Day 0 = onset",
        ha="center",
        fontsize=14,
        style="italic"
    )

    fig.text(
        0.5, 0.905,
        f"4-panel seasonal framework (DJF, MAM, JJA, SON) | Total onset sample: N={total_n}",
        ha="center",
        fontsize=11
    )

    # --- Legend
    legend_elements = [
        mpatches.Rectangle((0, 0), 1, 1, fill=False, edgecolor="k", linewidth=1.5, label="Valley of Mexico"),
        plt.Line2D([0], [0], marker="*", color="w", markerfacecolor="gold",
                   markeredgecolor="k", markersize=12, label="Mexico City (CDMX)")
    ]
    fig.legend(
        handles=legend_elements,
        loc="upper left",
        fontsize=11,
        frameon=False,
        bbox_to_anchor=(0.04, 0.91)
    )

    plt.savefig(out_fig_path, dpi=300)
    plt.close(fig)
    print(f"Saved: {out_fig_path}")

print("Stage-4-style seasonal plotting helpers are ready.")

Stage-4-style seasonal plotting helpers are ready.


### 6.2 PM10 seasonal day-0 composites

In [91]:
pm10_seasonal_composites = get_seasonal_composite_dict(
    z500_anom, u500_anom, v500_anom,
    pm10_onset_dates_by_season
)

pm10_fig_path = FIGURES_DIR / "stage5_PM10_seasonal_day0_composites_4panel.png"

plot_seasonal_composite_panel(
    pm10_seasonal_composites,
    pollutant_label="PM10",
    out_fig_path=pm10_fig_path,
    vector_step=6,
    contour_step=5
)

pm10_seasonal_summary = build_seasonal_summary_table(
    pm10_seasonal_composites,
    pollutant_label="PM10"
)

save_csv(pm10_seasonal_summary, "stage5_PM10_seasonal_day0_summary.csv", index=False)

print("Saved: stage5_PM10_seasonal_day0_summary.csv")
pm10_seasonal_summary

Saved: C:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 5\outputs\figures\stage5_PM10_seasonal_day0_composites_4panel.png
Saved: stage5_PM10_seasonal_day0_summary.csv


,pollutant,season,n_onset_days,max_abs_z500_anom,mean_abs_z500_anom,mean_wind_anom_mag
0,PM10,DJF,70,14.240988,4.260468,1.396046
1,PM10,MAM,53,32.756939,4.054671,1.501099
2,PM10,JJA,68,9.755809,2.886149,0.763328
3,PM10,SON,59,15.123734,5.767316,1.184687


### 6.3 PM2.5 seasonal day-0 composites

In [92]:
pm25_seasonal_composites = get_seasonal_composite_dict(
    z500_anom, u500_anom, v500_anom,
    pm25_onset_dates_by_season
)

pm25_fig_path = FIGURES_DIR / "stage5_PM25_seasonal_day0_composites_4panel.png"

plot_seasonal_composite_panel(
    pm25_seasonal_composites,
    pollutant_label="PM2.5",
    out_fig_path=pm25_fig_path,
    vector_step=6,
    contour_step=5
)

pm25_seasonal_summary = build_seasonal_summary_table(
    pm25_seasonal_composites,
    pollutant_label="PM2.5"
)

save_csv(pm25_seasonal_summary, "stage5_PM25_seasonal_day0_summary.csv", index=False)

print("Saved: stage5_PM25_seasonal_day0_summary.csv")
pm25_seasonal_summary

Saved: C:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 5\outputs\figures\stage5_PM25_seasonal_day0_composites_4panel.png
Saved: stage5_PM25_seasonal_day0_summary.csv


,pollutant,season,n_onset_days,max_abs_z500_anom,mean_abs_z500_anom,mean_wind_anom_mag
0,PM2.5,DJF,74,18.052292,5.133082,1.439092
1,PM2.5,MAM,63,20.283266,2.663425,1.297682
2,PM2.5,JJA,76,12.086612,4.246170,1.005242
3,PM2.5,SON,73,22.961071,6.560750,1.418074


### 6.4 Seasonal comparison diagnostics

In [93]:
seasonal_day0_comparison = pd.concat(
    [pm10_seasonal_summary, pm25_seasonal_summary],
    ignore_index=True
)

# Rank robustness within each pollutant using max abs Z500 anomaly
seasonal_day0_comparison["rank_by_max_abs_z500"] = (
    seasonal_day0_comparison
    .groupby("pollutant")["max_abs_z500_anom"]
    .rank(ascending=False, method="dense")
)

# Rank domain-mean absolute anomaly
seasonal_day0_comparison["rank_by_mean_abs_z500"] = (
    seasonal_day0_comparison
    .groupby("pollutant")["mean_abs_z500_anom"]
    .rank(ascending=False, method="dense")
)

In [94]:
# Add a compact interpretation column
def assign_visual_anchor(row):
    if row["season"] == "DJF":
        return "Dry benchmark"
    elif row["season"] == "MAM":
        return "Transition"
    elif row["season"] == "JJA":
        return "Wet contrast"
    else:
        return "Included for completeness"

In [95]:
seasonal_day0_comparison["seasonal_role"] = seasonal_day0_comparison.apply(assign_visual_anchor, axis=1)

save_csv(seasonal_day0_comparison, "seasonal_day0_comparison_diagnostics.csv", index=False)

print("Saved: seasonal_day0_comparison_diagnostics.csv")
seasonal_day0_comparison

Saved: seasonal_day0_comparison_diagnostics.csv


,pollutant,season,n_onset_days,max_abs_z500_anom,mean_abs_z500_anom,mean_wind_anom_mag,rank_by_max_abs_z500,rank_by_mean_abs_z500,seasonal_role
0,PM10,DJF,70,14.240988,4.260468,1.396046,3.0,2.0,Dry benchmark
1,PM10,MAM,53,32.756939,4.054671,1.501099,1.0,3.0,Transition
2,PM10,JJA,68,9.755809,2.886149,0.763328,4.0,4.0,Wet contrast
3,PM10,SON,59,15.123734,5.767316,1.184687,2.0,1.0,Included for completeness
4,PM2.5,DJF,74,18.052292,5.133082,1.439092,3.0,2.0,Dry benchmark
5,PM2.5,MAM,63,20.283266,2.663425,1.297682,2.0,4.0,Transition
6,PM2.5,JJA,76,12.086612,4.246170,1.005242,4.0,3.0,Wet contrast
7,PM2.5,SON,73,22.961071,6.560750,1.418074,1.0,1.0,Included for completeness


## 7. Seasonal lag composites around onset

### 7.1 Define the lag framework and helper functions

In [96]:
LAG_WINDOW = [-3, -2, -1, 0, 1, 2]
LAG_LABEL_MAP = {
    -3: "-3",
    -2: "-2",
    -1: "-1",
     0: "0",
     1: "+1",
     2: "+2",
}

In [ ]:
def shift_dates(dates: pd.DatetimeIndex, lag: int) -> pd.DatetimeIndex:
    """Shift onset dates by a lag in days."""
    return pd.DatetimeIndex(dates + pd.to_timedelta(lag, unit="D"))

def keep_only_available_dates(dates: pd.DatetimeIndex, available_time_index: pd.DatetimeIndex) -> pd.DatetimeIndex:
    """Keep only dates that exist in the available meteorological time axis."""
    return pd.DatetimeIndex(sorted(pd.Index(dates).intersection(pd.Index(available_time_index))))

def build_lag_composite_dict(z_anom, u_anom, v_anom, onset_dates_by_season, available_time_index):
    """
    Build seasonal lag composites for each season and lag.

    Returns a nested dictionary:
    out[season]["z"], out[season]["u"], out[season]["v"] -> DataArray with lag dimension
    out[season]["lag_counts"] -> DataFrame with N per lag
    """
    out = {}

    for season in SEASON_ORDER:
        onset_dates = onset_dates_by_season[season]

        z_list = []
        u_list = []
        v_list = []
        lag_counts_rows = []

        for lag in LAG_WINDOW:
            lag_dates = shift_dates(onset_dates, lag)
            lag_dates = keep_only_available_dates(lag_dates, available_time_index)

            lag_counts_rows.append({
                "season": season,
                "lag": lag,
                "n_valid_maps": len(lag_dates)
            })

            if len(lag_dates) == 0:
                # Create NaN fields if no valid dates are available
                z_comp = z_anom.isel(time=0).copy() * np.nan
                u_comp = u_anom.isel(time=0).copy() * np.nan
                v_comp = v_anom.isel(time=0).copy() * np.nan
            else:
                z_comp = z_anom.sel(time=lag_dates).mean("time")
                u_comp = u_anom.sel(time=lag_dates).mean("time")
                v_comp = v_anom.sel(time=lag_dates).mean("time")

            z_list.append(z_comp.expand_dims(lag=[lag]))
            u_list.append(u_comp.expand_dims(lag=[lag]))
            v_list.append(v_comp.expand_dims(lag=[lag]))

        out[season] = {
            "z": xr.concat(z_list, dim="lag"),
            "u": xr.concat(u_list, dim="lag"),
            "v": xr.concat(v_list, dim="lag"),
            "lag_counts": pd.DataFrame(lag_counts_rows),
            "n_onset_days": len(onset_dates)
        }

    return out

def plot_seasonal_lag_composite_6panel(
    lag_result,
    pollutant_label,
    season,
    out_fig_path,
    vector_step=6,
    contour_step=5
):
    """
    Export a 6-panel lag-composite figure (2x3) for one pollutant and one season.
    Style aligned with the Stage 4 figures.

    Shading: Z500 anomaly
    Contours: Z500 anomaly (positive solid / negative dashed)
    Vectors: U500/V500 anomalies
    """

    Z_comp = lag_result[season]["z"]
    U_comp = lag_result[season]["u"]
    V_comp = lag_result[season]["v"]
    lag_counts = lag_result[season]["lag_counts"]
    n_onsets = lag_result[season]["n_onset_days"]

    max_abs = float(np.nanmax(np.abs(Z_comp.values)))
    if not np.isfinite(max_abs) or max_abs == 0:
        max_abs = 1.0

    norm = TwoSlopeNorm(vcenter=0, vmin=-max_abs, vmax=max_abs)

    proj = ccrs.PlateCarree()
    fig, axes = plt.subplots(
        2, 3,
        figsize=(14.8, 9.8),
        dpi=250,
        subplot_kw={"projection": proj}
    )
    axes = axes.flatten()

    fig.subplots_adjust(
        left=0.06,
        right=0.94,
        bottom=0.16,
        top=0.82,
        wspace=0.08,
        hspace=0.24
    )

    pcm_ref = None

    for i, lag in enumerate(LAG_WINDOW):
        ax = axes[i]

        ax.set_extent(
            [
                DOMAIN_MAIN["lon_min"], DOMAIN_MAIN["lon_max"],
                DOMAIN_MAIN["lat_min"], DOMAIN_MAIN["lat_max"]
            ],
            crs=proj
        )

        ax.coastlines(resolution="50m", linewidth=0.5)
        ax.add_feature(cfeature.BORDERS, linewidth=0.4)
        ax.add_feature(cfeature.STATES.with_scale("50m"), linewidth=0.3)

        z = Z_comp.sel(lag=lag).values
        u = U_comp.sel(lag=lag).values
        v = V_comp.sel(lag=lag).values

        # --- Shading
        pcm = ax.pcolormesh(
            lon2d, lat2d, z,
            cmap="RdBu_r",
            norm=norm,
            shading="auto",
            transform=proj
        )
        pcm_ref = pcm

        # --- Contours
        lev = np.arange(-max_abs, max_abs + contour_step, contour_step)
        pos = lev[lev > 0]
        neg = lev[lev < 0]

        if len(pos) > 0:
            ax.contour(
                lon2d, lat2d, z,
                levels=pos,
                colors="k",
                linewidths=0.7,
                transform=proj
            )
        if len(neg) > 0:
            ax.contour(
                lon2d, lat2d, z,
                levels=neg,
                colors="k",
                linewidths=0.7,
                linestyles="--",
                transform=proj
            )

        # --- Vectors
        yy = np.arange(0, z.shape[0], vector_step)
        xx = np.arange(0, z.shape[1], vector_step)

        ax.quiver(
            lon2d[np.ix_(yy, xx)],
            lat2d[np.ix_(yy, xx)],
            u[np.ix_(yy, xx)],
            v[np.ix_(yy, xx)],
            transform=proj,
            color="black",
            scale=80,
            width=0.0022,
            headwidth=3.5,
            headlength=4.5,
            headaxislength=4.0,
            minlength=0.05,
            pivot="middle",
            zorder=4
        )

        # --- Valley of Mexico box
        rect = mpatches.Rectangle(
            (VOM_BOX[0], VOM_BOX[1]),
            VOM_BOX[2], VOM_BOX[3],
            fill=False,
            edgecolor="k",
            linewidth=1.0,
            linestyle="-",
            transform=proj
        )
        ax.add_patch(rect)

        # --- CDMX star
        ax.plot(
            CDMX_REF["lon"], CDMX_REF["lat"],
            marker="*",
            color="gold",
            markersize=9,
            markeredgecolor="k",
            transform=proj
        )

        # --- Subplot title
        n_valid = int(lag_counts.loc[lag_counts["lag"] == lag, "n_valid_maps"].iloc[0])
        ax.set_title(
            f"Lag {LAG_LABEL_MAP[lag]} (N={n_valid})",
            fontsize=12,
            weight="bold"
        )

        # --- Gridlines
        gl = ax.gridlines(
            draw_labels=True,
            linewidth=0.2,
            color="gray",
            alpha=0.5,
            linestyle="--"
        )
        gl.top_labels = False
        gl.right_labels = False

        # latitudes only on left column
        if i % 3 == 0:
            gl.left_labels = True
        else:
            gl.left_labels = False

        # longitudes on all subplots
        gl.bottom_labels = True

    # --- Bottom colorbar
    cbar_ax = fig.add_axes([0.30, 0.08, 0.40, 0.032])
    cb = fig.colorbar(pcm_ref, cax=cbar_ax, orientation="horizontal", extend="both")
    cb.set_label("Z500′ anomaly (m)", fontsize=12)
    cb.ax.tick_params(labelsize=10)

    # --- Titles
    fig.suptitle(
        f"{pollutant_label}: {season} lag composites around episode onset (2012–2024)",
        fontsize=20,
        weight="bold",
        y=0.975
    )

    fig.text(
        0.5, 0.93,
        "Shading/contours: Z500′ anomaly | Vectors: Δ(U′,V′) at 500 hPa | Day 0 = onset",
        ha="center",
        fontsize=14,
        style="italic"
    )

    fig.text(
        0.5, 0.907,
        f"6-panel lag sequence (−3 to +2 days) | Onset sample: N={n_onsets}",
        ha="center",
        fontsize=11
    )

    # --- Legend
    legend_elements = [
        mpatches.Rectangle((0, 0), 1, 1, fill=False, edgecolor="k", linewidth=1.5, label="Valley of Mexico"),
        plt.Line2D([0], [0], marker="*", color="w", markerfacecolor="gold",
                   markeredgecolor="k", markersize=12, label="Mexico City (CDMX)")
    ]
    fig.legend(
        handles=legend_elements,
        loc="upper left",
        fontsize=11,
        frameon=False,
        bbox_to_anchor=(0.03, 0.92)
    )

    plt.savefig(out_fig_path, dpi=300)
    plt.close(fig)
    print(f"Saved: {out_fig_path}")

def build_lag_summary_table(lag_result, pollutant_label):
    """
    Build a compact lag-summary table by season with:
    - onset N
    - max abs anomaly across all lags
    - mean abs anomaly at lag 0
    - mean abs anomaly at lag -1
    - mean abs anomaly at lag +1
    - mean wind anomaly magnitude at lag 0
    """
    rows = []

    for season in SEASON_ORDER:
        z_all = lag_result[season]["z"].values
        z_lag0 = lag_result[season]["z"].sel(lag=0).values
        z_lagm1 = lag_result[season]["z"].sel(lag=-1).values
        z_lagp1 = lag_result[season]["z"].sel(lag=1).values
        u_lag0 = lag_result[season]["u"].sel(lag=0).values
        v_lag0 = lag_result[season]["v"].sel(lag=0).values

        rows.append({
            "pollutant": pollutant_label,
            "season": season,
            "n_onset_days": lag_result[season]["n_onset_days"],
            "max_abs_z500_anylag": float(np.nanmax(np.abs(z_all))),
            "mean_abs_z500_lagm1": float(np.nanmean(np.abs(z_lagm1))),
            "mean_abs_z500_lag0": float(np.nanmean(np.abs(z_lag0))),
            "mean_abs_z500_lagp1": float(np.nanmean(np.abs(z_lagp1))),
            "mean_wind_anom_mag_lag0": float(np.nanmean(np.sqrt(u_lag0**2 + v_lag0**2))),
        })

    return pd.DataFrame(rows)

print("Lag framework and helper functions are ready.")

Lag framework and helper functions are ready.


### 7.2 PM10 seasonal lag composites

In [104]:
available_time_index = pd.DatetimeIndex(z500_anom["time"].values)

pm10_lag_result = build_lag_composite_dict(
    z500_anom, u500_anom, v500_anom,
    pm10_onset_dates_by_season,
    available_time_index
)

for season in SEASON_ORDER:
    out_fig = FIGURES_DIR / f"stage5_PM10_{season}_lag_composites_6panel.png"
    plot_seasonal_lag_composite_6panel(
        pm10_lag_result,
        pollutant_label="PM10",
        season=season,
        out_fig_path=out_fig,
        vector_step=6,
        contour_step=5
    )

pm10_lag_summary = build_lag_summary_table(pm10_lag_result, "PM10")
save_csv(pm10_lag_summary, "stage5_PM10_seasonal_lag_summary.csv", index=False)

print("Saved: stage5_PM10_seasonal_lag_summary.csv")
pm10_lag_summary

Saved: C:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 5\outputs\figures\stage5_PM10_DJF_lag_composites_6panel.png
Saved: C:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 5\outputs\figures\stage5_PM10_MAM_lag_composites_6panel.png
Saved: C:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 5\outputs\figures\stage5_PM10_JJA_lag_composites_6panel.png
Saved: C:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 5\outputs\figures\stage5_PM10_SON_lag_composites_6panel.png
Saved: stage5_PM10_seasonal_lag_summary.csv


,pollutant,season,n_onset_days,max_abs_z500_anylag,mean_abs_z500_lagm1,mean_abs_z500_lag0,mean_abs_z500_lagp1,mean_wind_anom_mag_lag0
0,PM10,DJF,70,23.223476,3.355952,4.260468,3.791568,1.396046
1,PM10,MAM,53,39.027454,6.494188,4.054671,3.796469,1.501099
2,PM10,JJA,68,9.755809,1.548498,2.886149,2.310589,0.763328
3,PM10,SON,59,24.903465,3.732033,5.767316,3.525907,1.184687


### 7.3 PM2.5 seasonal lag composites

In [105]:
pm25_lag_result = build_lag_composite_dict(
    z500_anom, u500_anom, v500_anom,
    pm25_onset_dates_by_season,
    available_time_index
)

for season in SEASON_ORDER:
    out_fig = FIGURES_DIR / f"stage5_PM25_{season}_lag_composites_6panel.png"
    plot_seasonal_lag_composite_6panel(
        pm25_lag_result,
        pollutant_label="PM2.5",
        season=season,
        out_fig_path=out_fig,
        vector_step=6,
        contour_step=5
    )

pm25_lag_summary = build_lag_summary_table(pm25_lag_result, "PM2.5")
save_csv(pm25_lag_summary, "stage5_PM25_seasonal_lag_summary.csv", index=False)

print("Saved: stage5_PM25_seasonal_lag_summary.csv")
pm25_lag_summary

Saved: C:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 5\outputs\figures\stage5_PM25_DJF_lag_composites_6panel.png
Saved: C:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 5\outputs\figures\stage5_PM25_MAM_lag_composites_6panel.png
Saved: C:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 5\outputs\figures\stage5_PM25_JJA_lag_composites_6panel.png
Saved: C:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 5\outputs\figures\stage5_PM25_SON_lag_composites_6panel.png
Saved: stage5_PM25_seasonal_lag_summary.csv


,pollutant,season,n_onset_days,max_abs_z500_anylag,mean_abs_z500_lagm1,mean_abs_z500_lag0,mean_abs_z500_lagp1,mean_wind_anom_mag_lag0
0,PM2.5,DJF,74,22.110413,4.575150,5.133082,5.544359,1.439092
1,PM2.5,MAM,63,28.017664,2.412752,2.663425,3.992866,1.297682
2,PM2.5,JJA,76,12.086612,3.699394,4.246170,3.020834,1.005242
3,PM2.5,SON,73,25.356720,4.770576,6.560750,5.629505,1.418074


### 7.4 Comparative lag summary

In [102]:
seasonal_lag_comparison = pd.concat(
    [pm10_lag_summary, pm25_lag_summary],
    ignore_index=True
)

# Add simple relative diagnostics
seasonal_lag_comparison["preconditioning_minus_onset"] = (
    seasonal_lag_comparison["mean_abs_z500_lagm1"] - seasonal_lag_comparison["mean_abs_z500_lag0"]
)

seasonal_lag_comparison["persistence_minus_onset"] = (
    seasonal_lag_comparison["mean_abs_z500_lagp1"] - seasonal_lag_comparison["mean_abs_z500_lag0"]
)

def lag_interpretation_anchor(row):
    if row["season"] == "DJF":
        return "Benchmark sequence"
    elif row["season"] == "MAM":
        return "Transition sequence"
    elif row["season"] == "JJA":
        return "Wet-season contrast"
    else:
        return "Included for completeness"

seasonal_lag_comparison["lag_role"] = seasonal_lag_comparison.apply(lag_interpretation_anchor, axis=1)

save_csv(seasonal_lag_comparison, "stage5_seasonal_lag_comparison_diagnostics.csv", index=False)

print("Saved: stage5_seasonal_lag_comparison_diagnostics.csv")
seasonal_lag_comparison

Saved: stage5_seasonal_lag_comparison_diagnostics.csv


,pollutant,season,n_onset_days,max_abs_z500_anylag,mean_abs_z500_lagm1,mean_abs_z500_lag0,mean_abs_z500_lagp1,mean_wind_anom_mag_lag0,preconditioning_minus_onset,persistence_minus_onset,lag_role
0,PM10,DJF,70,23.223476,3.355952,4.260468,3.791568,1.396046,-0.904517,-0.468901,Benchmark sequence
1,PM10,MAM,53,39.027454,6.494188,4.054671,3.796469,1.501099,2.439517,-0.258202,Transition sequence
2,PM10,JJA,68,9.755809,1.548498,2.886149,2.310589,0.763328,-1.337651,-0.575560,Wet-season contrast
3,PM10,SON,59,24.903465,3.732033,5.767316,3.525907,1.184687,-2.035283,-2.241409,Included for completeness
4,PM2.5,DJF,74,22.110413,4.575150,5.133082,5.544359,1.439092,-0.557933,0.411276,Benchmark sequence
5,PM2.5,MAM,63,28.017664,2.412752,2.663425,3.992866,1.297682,-0.250673,1.329441,Transition sequence
6,PM2.5,JJA,76,12.086612,3.699394,4.246170,3.020834,1.005242,-0.546776,-1.225336,Wet-season contrast
7,PM2.5,SON,73,25.356720,4.770576,6.560750,5.629505,1.418074,-1.790175,-0.931246,Included for completeness


### 7.5 Lagged seasonal diagnostics

In [117]:
# Regional box for large-scale Z500' summary (broader than the Valley of Mexico box)
SYNOPTIC_BOX = {
    "lon_min": -108,
    "lon_max": -92,
    "lat_min": 18,
    "lat_max": 30,
}

SEASON_COLORS = {
    "DJF": "#D37259",   # warm benchmark
    "MAM": "#71C2A0",   # transition
    "JJA": "#9999CC",   # wet-season contrast
    "SON": "#D8DA5D",   # drying-season context
}

In [118]:
def subset_box_2d_da(da: xr.DataArray, box: dict) -> xr.DataArray:
    """Subset a DataArray with 2D lon/lat coordinates using a rectangular mask."""
    lon2d = da["lon"]
    lat2d = da["lat"]

    mask = (
        (lon2d >= box["lon_min"]) & (lon2d <= box["lon_max"]) &
        (lat2d >= box["lat_min"]) & (lat2d <= box["lat_max"])
    )
    return da.where(mask, drop=True)

def build_lagged_seasonal_diagnostics_df(
    precip_ds,
    precip_var,
    z500_anom,
    u500_anom,
    v500_anom,
    onset_dates_by_season,
    available_time_index
):
    """
    Build a lagged seasonal diagnostics table with:
    - mean_precip_mmday_vom
    - mean_wspd500_anom_vom
    - mean_z500_anom_synoptic_box
    """
    rows = []

    # VOM subsets for precip / wind
    precip_vom_da = subset_box_2d_da(precip_ds[precip_var], {
        "lon_min": VOM_SW["lon"],
        "lon_max": VOM_NE["lon"],
        "lat_min": VOM_SW["lat"],
        "lat_max": VOM_NE["lat"],
    })

    u500_vom_da = subset_box_2d_da(u500_anom, {
        "lon_min": VOM_SW["lon"],
        "lon_max": VOM_NE["lon"],
        "lat_min": VOM_SW["lat"],
        "lat_max": VOM_NE["lat"],
    })

    v500_vom_da = subset_box_2d_da(v500_anom, {
        "lon_min": VOM_SW["lon"],
        "lon_max": VOM_NE["lon"],
        "lat_min": VOM_SW["lat"],
        "lat_max": VOM_NE["lat"],
    })

    # Broader synoptic box for Z500'
    z500_synoptic_da = subset_box_2d_da(z500_anom, SYNOPTIC_BOX)

    for season in SEASON_ORDER:
        onset_dates = onset_dates_by_season[season]

        for lag in LAG_WINDOW:
            lag_dates = shift_dates(onset_dates, lag)
            lag_dates = keep_only_available_dates(lag_dates, available_time_index)

            if len(lag_dates) == 0:
                mean_precip = np.nan
                mean_wspd500 = np.nan
                mean_z500 = np.nan
                n_valid = 0
            else:
                precip_comp = precip_vom_da.sel(time=lag_dates).mean("time")
                u_comp = u500_vom_da.sel(time=lag_dates).mean("time")
                v_comp = v500_vom_da.sel(time=lag_dates).mean("time")
                z_comp = z500_synoptic_da.sel(time=lag_dates).mean("time")

                mean_precip = float(precip_comp.mean(dim=("y", "x"), skipna=True).values)
                mean_wspd500 = float(np.sqrt(u_comp**2 + v_comp**2).mean(dim=("y", "x"), skipna=True).values)
                mean_z500 = float(z_comp.mean(dim=("y", "x"), skipna=True).values)
                n_valid = len(lag_dates)

            rows.append({
                "season": season,
                "lag": lag,
                "n_valid_maps": n_valid,
                "mean_precip_mmday_vom": mean_precip,
                "mean_wspd500_anom_vom": mean_wspd500,
                "mean_z500_anom_synoptic": mean_z500,
            })

    return pd.DataFrame(rows)

In [119]:
# Build one diagnostics table per pollutant (same meteorology framework, different onset dates)
pm10_lagged_diag = build_lagged_seasonal_diagnostics_df(
    precip_ds=precip_ds,
    precip_var=PRECIP_VAR,
    z500_anom=z500_anom,
    u500_anom=u500_anom,
    v500_anom=v500_anom,
    onset_dates_by_season=pm10_onset_dates_by_season,
    available_time_index=available_time_index
)

pm10_lagged_diag["pollutant"] = "PM10"

pm25_lagged_diag = build_lagged_seasonal_diagnostics_df(
    precip_ds=precip_ds,
    precip_var=PRECIP_VAR,
    z500_anom=z500_anom,
    u500_anom=u500_anom,
    v500_anom=v500_anom,
    onset_dates_by_season=pm25_onset_dates_by_season,
    available_time_index=available_time_index
)

pm25_lagged_diag["pollutant"] = "PM2.5"

seasonal_mech_lags = pd.concat([pm10_lagged_diag, pm25_lagged_diag], ignore_index=True)

save_csv(seasonal_mech_lags, "stage5_lagged_seasonal_mechanism_diagnostics.csv", index=False)

print("Saved: stage5_lagged_seasonal_mechanism_diagnostics.csv")
seasonal_mech_lags.head(12)

Saved: stage5_lagged_seasonal_mechanism_diagnostics.csv


,season,lag,n_valid_maps,mean_precip_mmday_vom,mean_wspd500_anom_vom,mean_z500_anom_synoptic,pollutant
0,DJF,-3,69,0.205415,1.577325,-6.077859,PM10
1,DJF,-2,69,0.303574,1.527301,-4.980507,PM10
2,DJF,-1,69,0.176782,1.152933,-0.392847,PM10
3,DJF,0,70,0.141896,1.172811,7.641245,PM10
4,DJF,1,69,0.210158,0.679048,6.586282,PM10
5,DJF,2,69,0.230510,0.282602,1.009171,PM10
6,MAM,-3,52,0.688475,2.655388,-15.543714,PM10
7,MAM,-2,52,0.556952,3.177059,-13.668835,PM10
8,MAM,-1,53,0.443523,2.159245,-8.742438,PM10
9,MAM,0,53,0.485335,0.959433,-3.570442,PM10


In [120]:
# ------------------------------------------------------------
# Plot compact lagged seasonal diagnostics
# ------------------------------------------------------------

def plot_lagged_seasonal_mechanism_panels(df_mech, pollutant_label, out_fig_path):
    """
    Three-panel lagged seasonal diagnostics figure for one pollutant:
    - precipitation over VOM
    - 500 hPa wind-anomaly magnitude over VOM
    - regional mean Z500' over a broader synoptic box
    comparing DJF, MAM, JJA, and SON
    """
    season_order = ["DJF", "MAM", "JJA", "SON"]

    fig, axes = plt.subplots(1, 3, figsize=(16.8, 5.0), dpi=250, sharex=True)

    metrics = [
        ("mean_precip_mmday_vom", "Precipitation\n(mm/day)", "Lagged precipitation over VOM"),
        ("mean_wspd500_anom_vom", "500 hPa wind anomaly\nmagnitude Δ(U′,V′) (m/s)", "Lagged 500 hPa flow over VOM"),
        ("mean_z500_anom_synoptic", "Regional mean Z500′ (m)", "Lagged regional Z500′"),
    ]

    sub = df_mech[df_mech["pollutant"] == pollutant_label].copy()

    for ax, (metric, ylabel, title) in zip(axes, metrics):
        for season in season_order:
            d = sub[sub["season"] == season].sort_values("lag")
            ax.plot(
                d["lag"],
                d[metric],
                marker="o",
                linewidth=2,
                label=season,
                color=SEASON_COLORS[season]
            )

        ax.axvline(0, color="k", linestyle="--", linewidth=1.0, alpha=0.7)
        ax.set_title(title, fontsize=10, fontweight="bold")
        ax.set_ylabel(ylabel)
        ax.grid(axis="y", linestyle="--", alpha=0.5)

    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(
        handles,
        labels,
        ncol=4,
        loc="lower center",
        bbox_to_anchor=(0.5, 0.04),
        frameon=True
    )

    axes[1].set_xlabel("Lag relative to episode onset (days)")
    axes[0].set_xticks(LAG_WINDOW)
    axes[0].set_xticklabels([LAG_LABEL_MAP[l] for l in LAG_WINDOW])

    fig.suptitle(
        f"{pollutant_label}: lagged seasonal diagnostics",
        fontsize=17,
        fontweight="bold",
        y=0.98
    )
    fig.text(
        0.5, 0.895,
        "Seasonal area-mean diagnostics | Day 0 = onset | DJF benchmark, MAM transition, JJA wet-season contrast, and SON drying-season context",
        ha="center",
        fontsize=11,
        style="italic"
    )

    fig.subplots_adjust(top=0.80, bottom=0.24, wspace=0.28)

    plt.savefig(out_fig_path, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: {out_fig_path}")

In [121]:
plot_lagged_seasonal_mechanism_panels(
    df_mech=seasonal_mech_lags,
    pollutant_label="PM10",
    out_fig_path=FIGURES_DIR / "stage5_PM10_lagged_seasonal_diagnostics.png"
)

plot_lagged_seasonal_mechanism_panels(
    df_mech=seasonal_mech_lags,
    pollutant_label="PM2.5",
    out_fig_path=FIGURES_DIR / "stage5_PM25_lagged_seasonal_diagnostics.png"
)

Saved: C:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 5\outputs\figures\stage5_PM10_lagged_seasonal_diagnostics.png
Saved: C:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 5\outputs\figures\stage5_PM25_lagged_seasonal_diagnostics.png


## 8. Significance testing on seasonal onset signals

### 8.1 Define the seasonal significance framework

In [122]:
SIGNIF_ALPHA = 0.05
SIGNIF_LAGS = [-1, 0]

In [129]:
def get_season_dates_from_pm(pm_df: pd.DataFrame, season: str) -> pd.DatetimeIndex:
    """Return all available PM dates for a given season."""
    return pd.DatetimeIndex(
        pm_df.loc[pm_df["season"] == season, "date"].sort_values().unique()
    )

def get_reference_dates_for_season(
    season: str,
    onset_dates: pd.DatetimeIndex,
    available_pm_dates: pd.DatetimeIndex
) -> pd.DatetimeIndex:
    """
    Seasonal non-onset reference days:
    all dates in the same season excluding onset days.
    """
    return pd.DatetimeIndex(sorted(pd.Index(available_pm_dates).difference(pd.Index(onset_dates))))

def get_balanced_dates(dates_a: pd.DatetimeIndex, dates_b: pd.DatetimeIndex) -> tuple[pd.DatetimeIndex, pd.DatetimeIndex]:
    """
    Balance sample sizes by taking the minimum N.
    Keeps chronological order and trims both sets to the same length.
    """
    n_use = min(len(dates_a), len(dates_b))
    return pd.DatetimeIndex(dates_a[:n_use]), pd.DatetimeIndex(dates_b[:n_use])

def welch_ttest_map(
    z_event: xr.DataArray,
    z_ref: xr.DataArray,
    u_event: xr.DataArray,
    u_ref: xr.DataArray,
    v_event: xr.DataArray,
    v_ref: xr.DataArray
) -> xr.Dataset:
    """
    Gridpoint-wise Welch t-test for Z500' onset vs reference maps.

    Returns:
    - z_event_mean
    - z_ref_mean
    - diff_mean          (Z500' onset - reference)
    - u_event_mean
    - u_ref_mean
    - u_diff_mean        (U500' onset - reference)
    - v_event_mean
    - v_ref_mean
    - v_diff_mean        (V500' onset - reference)
    - pval               (Welch test on Z500')
    - signif_mask
    """
    # --- Z500 test inputs
    z_event_vals = z_event.values   # shape: time, y, x
    z_ref_vals = z_ref.values

    tstat, pval = ttest_ind(
        z_event_vals,
        z_ref_vals,
        axis=0,
        equal_var=False,
        nan_policy="omit"
    )

    # --- Means for Z500
    z_event_mean = np.nanmean(z_event_vals, axis=0)
    z_ref_mean = np.nanmean(z_ref_vals, axis=0)
    z_diff_mean = z_event_mean - z_ref_mean

    # --- Means for U500
    u_event_mean = np.nanmean(u_event.values, axis=0)
    u_ref_mean = np.nanmean(u_ref.values, axis=0)
    u_diff_mean = u_event_mean - u_ref_mean

    # --- Means for V500
    v_event_mean = np.nanmean(v_event.values, axis=0)
    v_ref_mean = np.nanmean(v_ref.values, axis=0)
    v_diff_mean = v_event_mean - v_ref_mean

    signif_mask = pval < SIGNIF_ALPHA

    return xr.Dataset(
        data_vars={
            "z_event_mean": (("y", "x"), z_event_mean),
            "z_ref_mean": (("y", "x"), z_ref_mean),
            "diff_mean": (("y", "x"), z_diff_mean),

            "u_event_mean": (("y", "x"), u_event_mean),
            "u_ref_mean": (("y", "x"), u_ref_mean),
            "u_diff_mean": (("y", "x"), u_diff_mean),

            "v_event_mean": (("y", "x"), v_event_mean),
            "v_ref_mean": (("y", "x"), v_ref_mean),
            "v_diff_mean": (("y", "x"), v_diff_mean),

            "pval": (("y", "x"), pval),
            "signif_mask": (("y", "x"), signif_mask.astype(int)),
        },
        coords={
            "y": z_event["y"],
            "x": z_event["x"],
            "lat": (("y", "x"), z_event["lat"].values),
            "lon": (("y", "x"), z_event["lon"].values),
        }
    )

def build_significance_results_for_pollutant(
    z500_anom,
    u500_anom,
    v500_anom,
    onset_dates_by_season,
    pm_daily_with_season
):
    """
    Build significance results for one pollutant for:
    - lag 0
    - lag -1
    comparing onset vs seasonal non-onset reference days.

    Welch significance is computed on Z500'.
    U500'/V500' are stored as onset - reference mean differences for plotting.
    """
    results = {}

    for season in SEASON_ORDER:
        onset_dates = onset_dates_by_season[season]
        all_season_dates = get_season_dates_from_pm(pm_daily_with_season, season)
        ref_dates = get_reference_dates_for_season(season, onset_dates, all_season_dates)

        season_results = {}

        for lag in SIGNIF_LAGS:
            event_dates = keep_only_available_dates(
                shift_dates(onset_dates, lag),
                available_time_index
            )
            ref_lag_dates = keep_only_available_dates(
                shift_dates(ref_dates, lag),
                available_time_index
            )

            event_dates_bal, ref_dates_bal = get_balanced_dates(event_dates, ref_lag_dates)

            # Z500 samples
            z_event = z500_anom.sel(time=event_dates_bal)
            z_ref = z500_anom.sel(time=ref_dates_bal)

            # U500 samples
            u_event = u500_anom.sel(time=event_dates_bal)
            u_ref = u500_anom.sel(time=ref_dates_bal)

            # V500 samples
            v_event = v500_anom.sel(time=event_dates_bal)
            v_ref = v500_anom.sel(time=ref_dates_bal)

            ds_sig = welch_ttest_map(
                z_event=z_event,
                z_ref=z_ref,
                u_event=u_event,
                u_ref=u_ref,
                v_event=v_event,
                v_ref=v_ref
            )

            ds_sig = ds_sig.assign_coords(lag=lag)
            ds_sig = ds_sig.assign_attrs({
                "season": season,
                "n_event": len(event_dates_bal),
                "n_ref": len(ref_dates_bal),
                "alpha": SIGNIF_ALPHA,
            })

            season_results[lag] = ds_sig

        results[season] = season_results

    return results

print("Seasonal significance framework is ready.")

Seasonal significance framework is ready.


### 8.2 Day-0 significance maps

In [ ]:
def plot_significance_panel(
    signif_results,
    pollutant_label,
    lag,
    out_fig_path,
    contour_step=5,
    stipple_thin=5,
    vector_step=8
):
    """
    4-panel seasonal significance figure for one pollutant and one lag.

    Shading: onset - reference mean Z500'
    Contours: same difference field
    Vectors: onset - reference mean U500'/V500'
    Stippling: p < alpha from Welch test on Z500'
    """

    # collect all seasonal diff fields to define a symmetric color range
    max_abs = max(
        float(np.nanmax(np.abs(signif_results[season][lag]["diff_mean"].values)))
        for season in SEASON_ORDER
    )
    if not np.isfinite(max_abs) or max_abs == 0:
        max_abs = 1.0

    norm = TwoSlopeNorm(vcenter=0, vmin=-max_abs, vmax=max_abs)

    proj = ccrs.PlateCarree()
    fig, axes = plt.subplots(
        2, 2,
        figsize=(13.5, 10.5),
        dpi=250,
        subplot_kw={"projection": proj}
    )
    axes = axes.flatten()

    fig.subplots_adjust(
        left=0.07,
        right=0.93,
        bottom=0.16,
        top=0.81,
        wspace=0.10,
        hspace=0.20
    )

    pcm_ref = None

    for i, season in enumerate(SEASON_ORDER):
        ax = axes[i]
        ds = signif_results[season][lag]

        diff = ds["diff_mean"].values
        u_diff = ds["u_diff_mean"].values
        v_diff = ds["v_diff_mean"].values
        pmask = ds["signif_mask"].values.astype(bool)
        lon = ds["lon"].values
        lat = ds["lat"].values

        ax.set_extent(
            [
                DOMAIN_MAIN["lon_min"], DOMAIN_MAIN["lon_max"],
                DOMAIN_MAIN["lat_min"], DOMAIN_MAIN["lat_max"]
            ],
            crs=proj
        )

        ax.coastlines(resolution="50m", linewidth=0.5)
        ax.add_feature(cfeature.BORDERS, linewidth=0.4)
        ax.add_feature(cfeature.STATES.with_scale("50m"), linewidth=0.3)

        # --- Shading
        pcm = ax.pcolormesh(
            lon, lat, diff,
            cmap="RdBu_r",
            norm=norm,
            shading="auto",
            transform=proj
        )
        pcm_ref = pcm

        # --- Contours
        lev = np.arange(-max_abs, max_abs + contour_step, contour_step)
        pos = lev[lev > 0]
        neg = lev[lev < 0]

        if len(pos) > 0:
            ax.contour(
                lon, lat, diff,
                levels=pos,
                colors="k",
                linewidths=0.7,
                transform=proj
            )

        if len(neg) > 0:
            ax.contour(
                lon, lat, diff,
                levels=neg,
                colors="k",
                linewidths=0.7,
                linestyles="--",
                transform=proj
            )

        # --- Vectors: onset - reference mean U'/V'
        yy = np.arange(0, diff.shape[0], vector_step)
        xx = np.arange(0, diff.shape[1], vector_step)

        ax.quiver(
            lon[np.ix_(yy, xx)],
            lat[np.ix_(yy, xx)],
            u_diff[np.ix_(yy, xx)],
            v_diff[np.ix_(yy, xx)],
            transform=proj,
            color="black",
            scale=80,
            width=0.0022,
            headwidth=3.5,
            headlength=4.5,
            headaxislength=4.0,
            minlength=0.05,
            pivot="middle",
            zorder=4
        )

        # --- Stippling (thinned, like Stage 3)
        yy_sig, xx_sig = np.where(pmask)
        yy_sig = yy_sig[::stipple_thin]
        xx_sig = xx_sig[::stipple_thin]

        if len(yy_sig) > 0:
            ax.scatter(
                lon[yy_sig, xx_sig],
                lat[yy_sig, xx_sig],
                s=2,
                c="k",
                alpha=0.25,
                transform=proj,
                zorder=5
            )

        # --- Valley of Mexico box
        rect = mpatches.Rectangle(
            (VOM_BOX[0], VOM_BOX[1]),
            VOM_BOX[2], VOM_BOX[3],
            fill=False,
            edgecolor="k",
            linewidth=1.0,
            linestyle="-",
            transform=proj
        )
        ax.add_patch(rect)

        # --- CDMX star
        ax.plot(
            CDMX_REF["lon"], CDMX_REF["lat"],
            marker="*",
            color="gold",
            markersize=9,
            markeredgecolor="k",
            transform=proj
        )

        n_event = ds.attrs["n_event"]
        n_ref = ds.attrs["n_ref"]

        if n_event == n_ref:
            title_txt = f"{season} (N={n_event})"
        else:
            title_txt = f"{season} (N={n_event}, ref={n_ref})"

        ax.set_title(
            title_txt,
            fontsize=14,
            weight="bold"
        )

        gl = ax.gridlines(
            draw_labels=True,
            linewidth=0.2,
            color="gray",
            alpha=0.5,
            linestyle="--"
        )
        gl.top_labels = False
        gl.right_labels = False

        # latitudes only on left column
        if i % 2 == 0:
            gl.left_labels = True
        else:
            gl.left_labels = False

        # longitudes on all panels
        gl.bottom_labels = True

    # --- Bottom colorbar
    cbar_ax = fig.add_axes([0.30, 0.07, 0.40, 0.028])
    cb = fig.colorbar(pcm_ref, cax=cbar_ax, orientation="horizontal", extend="both")
    cb.set_label("Z500′ difference (onset − reference) (m)", fontsize=12)
    cb.ax.tick_params(labelsize=10)

    lag_label = "day 0" if lag == 0 else f"lag {lag}"

    fig.suptitle(
        f"{pollutant_label}: seasonal significance maps at {lag_label} (2012–2024)",
        fontsize=20,
        weight="bold",
        y=0.975
    )
    fig.text(
        0.5, 0.928,
        "Shading/contours: onset − seasonal non-onset reference | Vectors: Δ(U′,V′) onset − reference | Stippling: p < 0.05 (Welch test)",
        ha="center",
        fontsize=14,
        style="italic"
    )
    fig.text(
        0.5, 0.9045,
        "N denotes the balanced sample size used for both onset and seasonal non-onset reference days",
        ha="center",
        fontsize=11
    )

    # --- Legend
    legend_elements = [
        mpatches.Rectangle((0, 0), 1, 1, fill=False, edgecolor="k", linewidth=1.5, label="Valley of Mexico"),
        plt.Line2D(
            [0], [0],
            marker="*",
            color="w",
            markerfacecolor="gold",
            markeredgecolor="k",
            markersize=12,
            label="Mexico City (CDMX)"
        )
    ]
    fig.legend(
        handles=legend_elements,
        loc="upper left",
        fontsize=11,
        frameon=False,
        bbox_to_anchor=(0.04, 0.905)
    )

    plt.savefig(out_fig_path, dpi=300)
    plt.close(fig)
    print(f"Saved: {out_fig_path}")

In [131]:
# Build results
pm10_signif_results = build_significance_results_for_pollutant(
    z500_anom=z500_anom,
    u500_anom=u500_anom,
    v500_anom=v500_anom,
    onset_dates_by_season=pm10_onset_dates_by_season,
    pm_daily_with_season=pm_daily_with_season
)

pm25_signif_results = build_significance_results_for_pollutant(
    z500_anom=z500_anom,
    u500_anom=u500_anom,
    v500_anom=v500_anom,
    onset_dates_by_season=pm25_onset_dates_by_season,
    pm_daily_with_season=pm_daily_with_season
)

In [156]:
# Plot lag 0 significance
plot_significance_panel(
    signif_results=pm10_signif_results,
    pollutant_label="PM10",
    lag=0,
    out_fig_path=FIGURES_DIR / "stage5_PM10_seasonal_significance_day0.png"
)

plot_significance_panel(
    signif_results=pm25_signif_results,
    pollutant_label="PM2.5",
    lag=0,
    out_fig_path=FIGURES_DIR / "stage5_PM25_seasonal_significance_day0.png"
)

Saved: C:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 5\outputs\figures\stage5_PM10_seasonal_significance_day0.png
Saved: C:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 5\outputs\figures\stage5_PM25_seasonal_significance_day0.png


### 8.3 Selected precursor significance maps

In [157]:
plot_significance_panel(
    signif_results=pm10_signif_results,
    pollutant_label="PM10",
    lag=-1,
    out_fig_path=FIGURES_DIR / "stage5_PM10_seasonal_significance_lagm1.png"
)

plot_significance_panel(
    signif_results=pm25_signif_results,
    pollutant_label="PM2.5",
    lag=-1,
    out_fig_path=FIGURES_DIR / "stage5_PM25_seasonal_significance_lagm1.png"
)

Saved: C:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 5\outputs\figures\stage5_PM10_seasonal_significance_lagm1.png
Saved: C:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 5\outputs\figures\stage5_PM25_seasonal_significance_lagm1.png


### 8.4 Fraction of significant area and ranking

In [144]:
def signif_fraction_from_ds(ds_sig: xr.Dataset) -> float:
    """Fraction of gridcells with p < alpha."""
    mask = ds_sig["signif_mask"].values.astype(bool)
    return float(np.nanmean(mask))

In [145]:
rows = []

for pollutant_label, results in [("PM10", pm10_signif_results), ("PM2.5", pm25_signif_results)]:
    for season in SEASON_ORDER:
        for lag in SIGNIF_LAGS:
            ds = results[season][lag]
            rows.append({
                "pollutant": pollutant_label,
                "season": season,
                "lag": lag,
                "n_event": ds.attrs["n_event"],
                "n_ref": ds.attrs["n_ref"],
                "frac_significant_area": signif_fraction_from_ds(ds),
            })

seasonal_signif_summary = pd.DataFrame(rows)

seasonal_signif_summary["rank_within_pollutant_lag"] = (
    seasonal_signif_summary
    .groupby(["pollutant", "lag"])["frac_significant_area"]
    .rank(ascending=False, method="dense")
)

save_csv(seasonal_signif_summary, "stage5_seasonal_significance_summary.csv", index=False)

print("Saved: stage5_seasonal_significance_summary.csv")
seasonal_signif_summary

Saved: stage5_seasonal_significance_summary.csv


,pollutant,season,lag,n_event,n_ref,frac_significant_area,rank_within_pollutant_lag
0,PM10,DJF,-1,69,69,0.481667,3.0
1,PM10,DJF,0,70,70,0.576111,2.0
2,PM10,MAM,-1,53,53,0.518968,2.0
3,PM10,MAM,0,53,53,0.542698,3.0
4,PM10,JJA,-1,68,68,0.704524,1.0
5,PM10,JJA,0,68,68,0.625397,1.0
6,PM10,SON,-1,57,57,0.141667,4.0
7,PM10,SON,0,59,59,0.162302,4.0
8,PM2.5,DJF,-1,73,73,0.511190,3.0
9,PM2.5,DJF,0,74,74,0.604921,1.0


### 8.5 Interpretation support table

In [146]:
def assign_signif_note(row):
    if row["frac_significant_area"] >= 0.20:
        return "Strong spatial support"
    elif row["frac_significant_area"] >= 0.10:
        return "Moderate spatial support"
    elif row["frac_significant_area"] >= 0.05:
        return "Limited but non-negligible support"
    else:
        return "Weak spatial support"

In [147]:
seasonal_signif_notes = seasonal_signif_summary.copy()
seasonal_signif_notes["interpretation_note"] = seasonal_signif_notes.apply(assign_signif_note, axis=1)

save_csv(seasonal_signif_notes, "stage5_seasonal_significance_notes.csv", index=False)

print("Saved: stage5_seasonal_significance_notes.csv")
seasonal_signif_notes

Saved: stage5_seasonal_significance_notes.csv


,pollutant,season,lag,n_event,n_ref,frac_significant_area,rank_within_pollutant_lag,interpretation_note
0,PM10,DJF,-1,69,69,0.481667,3.0,Strong spatial support
1,PM10,DJF,0,70,70,0.576111,2.0,Strong spatial support
2,PM10,MAM,-1,53,53,0.518968,2.0,Strong spatial support
3,PM10,MAM,0,53,53,0.542698,3.0,Strong spatial support
4,PM10,JJA,-1,68,68,0.704524,1.0,Strong spatial support
5,PM10,JJA,0,68,68,0.625397,1.0,Strong spatial support
6,PM10,SON,-1,57,57,0.141667,4.0,Moderate spatial support
7,PM10,SON,0,59,59,0.162302,4.0,Moderate spatial support
8,PM2.5,DJF,-1,73,73,0.511190,3.0,Strong spatial support
9,PM2.5,DJF,0,74,74,0.604921,1.0,Strong spatial support


## 9. Focused physical consistency check

### 9.1 Scope note and focused support design

In [158]:
scope_note = pd.DataFrame({
    "component": [
        "Purpose of this section",
        "Role in thesis narrative",
        "Primary support variable",
        "Additional dynamical context",
        "Excluded from core analysis"
    ],
    "decision": [
        "Focused physical consistency support",
        "Support only, not a new analytical branch",
        "Precipitation over the Valley of Mexico",
        "Use already-derived 500 hPa circulation results from previous sections",
        "850 hPa and SH850 diagnostics"
    ]
})

save_csv(scope_note, "stage5_focused_physical_scope_note.csv", index=False)

scope_note

,component,decision
0,Purpose of this section,Focused physical consistency support
1,Role in thesis narrative,"Support only, not a new analytical branch"
2,Primary support variable,Precipitation over the Valley of Mexico
3,Additional dynamical context,Use already-derived 500 hPa circulation result...
4,Excluded from core analysis,850 hPa and SH850 diagnostics


### 9.2 Seasonal precipitation support

In [160]:
seasonal_precip_support = (
    precip_vom_daily
    .groupby("season")["precip_vom_mean"]
    .agg(
        precip_mean_mmday="mean",
        precip_median_mmday="median",
        precip_min_mmday="min",
        precip_max_mmday="max"
    )
    .reset_index()
)

seasonal_precip_support["season"] = pd.Categorical(
    seasonal_precip_support["season"],
    categories=SEASON_ORDER,
    ordered=True
)

seasonal_precip_support = (
    seasonal_precip_support
    .sort_values("season")
    .reset_index(drop=True)
)

save_csv(seasonal_precip_support, "stage5_seasonal_precip_support.csv", index=False)

print("Saved: stage5_seasonal_precip_support.csv")
seasonal_precip_support

Saved: stage5_seasonal_precip_support.csv


,season,precip_mean_mmday,precip_median_mmday,precip_min_mmday,precip_max_mmday
0,DJF,0.354234,0.083816,0.0,14.078870
1,MAM,1.163150,0.360832,0.0,19.758375
2,JJA,4.688356,3.979954,0.0,28.894218
3,SON,2.887631,1.504533,0.0,21.776615


### 9.3 Focused February–May–July support table

In [161]:
focused_months = [2, 5, 7]

focused_precip_support = (
    precip_vom_daily.loc[precip_vom_daily["month"].isin(focused_months)]
    .groupby("month")["precip_vom_mean"]
    .agg(
        precip_mean_mmday="mean",
        precip_median_mmday="median",
        precip_min_mmday="min",
        precip_max_mmday="max"
    )
    .reset_index()
)

focused_precip_support["month_name"] = focused_precip_support["month"].apply(month_name_from_number)

focused_role_map = {
    2: "Dry benchmark",
    5: "Transition case",
    7: "Wet-season contrast"
}
focused_precip_support["focused_role"] = focused_precip_support["month"].map(focused_role_map)

focused_precip_support = focused_precip_support[
    [
        "month",
        "month_name",
        "focused_role",
        "precip_mean_mmday",
        "precip_median_mmday",
        "precip_min_mmday",
        "precip_max_mmday"
    ]
]

save_csv(focused_precip_support, "stage5_feb_may_jul_precip_support.csv", index=False)

print("Saved: stage5_feb_may_jul_precip_support.csv")
focused_precip_support

Saved: stage5_feb_may_jul_precip_support.csv


,month,month_name,focused_role,precip_mean_mmday,precip_median_mmday,precip_min_mmday,precip_max_mmday
0,2,February,Dry benchmark,0.333571,0.058783,0.0,14.078870
1,5,May,Transition case,2.015312,1.249055,0.0,14.135362
2,7,July,Wet-season contrast,4.648746,4.011996,0.0,21.003939


### 9.4 Interpretation support

In [162]:
physical_support_notes = pd.DataFrame({
    "focus": [
        "DJF",
        "MAM",
        "JJA",
        "SON",
        "February",
        "May",
        "July"
    ],
    "interpretation_support": [
        "Dry seasonal background supports the benchmark interpretation of winter onset signals.",
        "Intermediate precipitation supports the transition-season interpretation rather than a pure winter-type benchmark.",
        "Wet background supports the interpretation of JJA as a wet-season contrast with altered or weakened PM10 circulation organization.",
        "Lower precipitation than JJA but wetter conditions than DJF support SON as a drying-season context or partial reorganization period rather than a benchmark season.",
        "Very low precipitation reinforces February as the clearest dry benchmark month.",
        "Moderate precipitation supports May as a genuine pre-monsoon transition case.",
        "Higher precipitation reinforces July as a wet-season contrast rather than a simple extension of the dry-season story."
    ]
})

save_csv(physical_support_notes, "stage5_physical_support_notes.csv", index=False)

print("Saved: stage5_physical_support_notes.csv")
physical_support_notes

Saved: stage5_physical_support_notes.csv


,focus,interpretation_support
0,DJF,Dry seasonal background supports the benchmark...
1,MAM,Intermediate precipitation supports the transi...
2,JJA,Wet background supports the interpretation of ...
3,SON,Lower precipitation than JJA but wetter condit...
4,February,Very low precipitation reinforces February as ...
5,May,Moderate precipitation supports May as a genui...
6,July,Higher precipitation reinforces July as a wet-...


## 10. ENSO modulation

### 10.1 Load and classify ENSO phases

#### 10.1.1 Load CPC RONI ASCII table

In [166]:
RONI_ASCII_URL = "https://www.cpc.ncep.noaa.gov/data/indices/RONI.ascii.txt"

resp = requests.get(RONI_ASCII_URL, timeout=30)
resp.raise_for_status()

roni_text = resp.text
print(roni_text[:500])

SEAS   YR  ANOM
DJF  1950 -1.47
JFM  1950 -1.27
FMA  1950 -1.11
MAM  1950 -1.09
AMJ  1950 -1.00
MJJ  1950 -0.71
JJA  1950 -0.41
JAS  1950 -0.24
ASO  1950 -0.22
SON  1950 -0.29
OND  1950 -0.41
NDJ  1950 -0.56
DJF  1951 -0.50
JFM  1951 -0.17
FMA  1951  0.25
MAM  1951  0.60
AMJ  1951  0.64
MJJ  1951  0.69
JJA  1951  0.68
JAS  1951  0.81
ASO  1951  0.87
SON  1951  0.98
OND  1951  0.91
NDJ  1951  0.80
DJF  1952  0.58
JFM  1952  0.44
FMA  1952  0.42
MAM  1952  0.44
AMJ  1952  0.31
MJJ  1952  0.07
JJA 


#### 10.1.2 Parse the historical RONI table

In [167]:
# The CPC ASCII file is whitespace-delimited with columns like:
# YR MON ANOM
# or sometimes a seasonal wide table depending on CPC formatting updates.
# We first try the common monthly/seasonal ASCII parse.

roni_raw = pd.read_csv(
    StringIO(roni_text),
    sep=r"\s+",
    engine="python"
)

print("Parsed columns:", list(roni_raw.columns))
roni_raw.head(12)

Parsed columns: ['SEAS', 'YR', 'ANOM']


,SEAS,YR,ANOM
0,DJF,1950,-1.47
1,JFM,1950,-1.27
2,FMA,1950,-1.11
3,MAM,1950,-1.09
4,AMJ,1950,-1.00
5,MJJ,1950,-0.71
6,JJA,1950,-0.41
7,JAS,1950,-0.24
8,ASO,1950,-0.22
9,SON,1950,-0.29


#### 10.1.3 Reshape to canonical seasonal long format

In [169]:
canonical_seasons = ["DJF", "MAM", "JJA", "SON"]

# CPC RONI ASCII is already in long seasonal format:
# SEAS | YR | ANOM
if {"SEAS", "YR", "ANOM"}.issubset(set(roni_raw.columns)):
    roni_long = (
        roni_raw.rename(columns={
            "SEAS": "season",
            "YR": "year",
            "ANOM": "roni"
        })[["year", "season", "roni"]]
        .copy()
    )
else:
    raise ValueError(
        f"Unrecognized RONI file structure. Columns found: {list(roni_raw.columns)}"
    )

# Keep only thesis period
roni_long["year"] = pd.to_numeric(roni_long["year"], errors="coerce")
roni_long["roni"] = pd.to_numeric(roni_long["roni"], errors="coerce")

roni_long = roni_long.dropna(subset=["year", "season", "roni"]).copy()
roni_long["year"] = roni_long["year"].astype(int)

roni_long = roni_long.loc[
    (roni_long["year"] >= 2012) & (roni_long["year"] <= 2024)
].copy()

# Keep only canonical seasons used in the thesis
roni_long = roni_long.loc[roni_long["season"].isin(canonical_seasons)].copy()

roni_long["season"] = pd.Categorical(
    roni_long["season"],
    categories=SEASON_ORDER,
    ordered=True
)

roni_long = roni_long.sort_values(["year", "season"]).reset_index(drop=True)

print("Canonical RONI table for 2012–2024:")
roni_long.head(12)

Canonical RONI table for 2012–2024:


,year,season,roni
0,2012,DJF,-0.82
1,2012,MAM,-0.47
2,2012,JJA,0.34
3,2012,SON,0.23
4,2013,DJF,-0.63
5,2013,MAM,-0.43
6,2013,JJA,-0.38
7,2013,SON,-0.22
8,2014,DJF,-0.49
9,2014,MAM,-0.05


#### 10.1.4 Classify ENSO phases from RONI

In [170]:
ENSO_THRESHOLD = 0.5

In [171]:
def classify_enso_phase(roni_value, threshold=ENSO_THRESHOLD):
    if pd.isna(roni_value):
        return np.nan
    elif roni_value >= threshold:
        return "El Niño"
    elif roni_value <= -threshold:
        return "La Niña"
    else:
        return "Neutral"

In [172]:
roni_long["enso_phase"] = roni_long["roni"].apply(classify_enso_phase)

phase_short_map = {
    "El Niño": "EN",
    "La Niña": "LN",
    "Neutral": "NEU"
}
roni_long["enso_phase_short"] = roni_long["enso_phase"].map(phase_short_map)

roni_long.head(12)

,year,season,roni,enso_phase,enso_phase_short
0,2012,DJF,-0.82,La Niña,LN
1,2012,MAM,-0.47,Neutral,NEU
2,2012,JJA,0.34,Neutral,NEU
3,2012,SON,0.23,Neutral,NEU
4,2013,DJF,-0.63,La Niña,LN
5,2013,MAM,-0.43,Neutral,NEU
6,2013,JJA,-0.38,Neutral,NEU
7,2013,SON,-0.22,Neutral,NEU
8,2014,DJF,-0.49,Neutral,NEU
9,2014,MAM,-0.05,Neutral,NEU


#### 10.1.5 Create season-year lookup table

In [173]:
enso_lookup = roni_long.copy()
enso_lookup["season_year"] = (
    enso_lookup["year"].astype(str) + "_" + enso_lookup["season"].astype(str)
)

save_csv(enso_lookup, "stage5_enso_roni_lookup_2012_2024.csv", index=False)

print("Saved: stage5_enso_roni_lookup_2012_2024.csv")
enso_lookup

Saved: stage5_enso_roni_lookup_2012_2024.csv


,year,season,roni,enso_phase,enso_phase_short,season_year
0,2012,DJF,-0.82,La Niña,LN,2012_DJF
1,2012,MAM,-0.47,Neutral,NEU,2012_MAM
2,2012,JJA,0.34,Neutral,NEU,2012_JJA
3,2012,SON,0.23,Neutral,NEU,2012_SON
4,2013,DJF,-0.63,La Niña,LN,2013_DJF
5,2013,MAM,-0.43,Neutral,NEU,2013_MAM
6,2013,JJA,-0.38,Neutral,NEU,2013_JJA
7,2013,SON,-0.22,Neutral,NEU,2013_SON
8,2014,DJF,-0.49,Neutral,NEU,2014_DJF
9,2014,MAM,-0.05,Neutral,NEU,2014_MAM


#### 10.1.6 Quick phase counts check

In [174]:
enso_phase_counts = (
    enso_lookup.groupby(["season", "enso_phase"], as_index=False)
    .size()
    .rename(columns={"size": "n_season_years"})
)

save_csv(enso_phase_counts, "stage5_enso_phase_counts_by_season.csv", index=False)

print("Saved: stage5_enso_phase_counts_by_season.csv")
enso_phase_counts

Saved: stage5_enso_phase_counts_by_season.csv


,season,enso_phase,n_season_years
0,DJF,El Niño,3
1,DJF,La Niña,7
2,DJF,Neutral,3
3,MAM,El Niño,3
4,MAM,La Niña,3
5,MAM,Neutral,7
6,JJA,El Niño,2
7,JJA,La Niña,5
8,JJA,Neutral,6
9,SON,El Niño,3


### 10.2 Event frequency by ENSO phase

#### 10.2.1 Attach ENSO phases to seasonal onset tables

In [181]:
ENSO_PHASE_ORDER = ["El Niño", "Neutral", "La Niña"]

# Keep only the merge keys we need
enso_merge = enso_lookup[["year", "season", "roni", "enso_phase", "enso_phase_short", "season_year"]].copy()

# -----------------------------
# PM10 onset table with ENSO phase
# -----------------------------
pm10_onset_enso = pm10_onset_framework.copy()

# ENSO year logic:
# - MAM/JJA/SON use the same calendar year
# - DJF December must be assigned to the following year's DJF
pm10_onset_enso["enso_year"] = pm10_onset_enso["year"]

pm10_onset_enso.loc[
    (pm10_onset_enso["season"] == "DJF") & (pm10_onset_enso["month"] == 12),
    "enso_year"
] = (
    pm10_onset_enso.loc[
        (pm10_onset_enso["season"] == "DJF") & (pm10_onset_enso["month"] == 12),
        "enso_year"
    ] + 1
)

pm10_onset_enso["season_year"] = (
    pm10_onset_enso["enso_year"].astype(str) + "_" + pm10_onset_enso["season"].astype(str)
)

pm10_onset_enso = pm10_onset_enso.merge(
    enso_merge,
    left_on=["enso_year", "season", "season_year"],
    right_on=["year", "season", "season_year"],
    how="left",
    validate="many_to_one",
    suffixes=("", "_enso")
)

# Clean duplicate merge year column
if "year_enso" in pm10_onset_enso.columns:
    pm10_onset_enso = pm10_onset_enso.drop(columns=["year_enso"])

pm10_onset_enso["pollutant"] = "PM10"

# -----------------------------
# PM2.5 onset table with ENSO phase
# -----------------------------
pm25_onset_enso = pm25_onset_framework.copy()

pm25_onset_enso["enso_year"] = pm25_onset_enso["year"]

pm25_onset_enso.loc[
    (pm25_onset_enso["season"] == "DJF") & (pm25_onset_enso["month"] == 12),
    "enso_year"
] = (
    pm25_onset_enso.loc[
        (pm25_onset_enso["season"] == "DJF") & (pm25_onset_enso["month"] == 12),
        "enso_year"
    ] + 1
)

pm25_onset_enso["season_year"] = (
    pm25_onset_enso["enso_year"].astype(str) + "_" + pm25_onset_enso["season"].astype(str)
)

pm25_onset_enso = pm25_onset_enso.merge(
    enso_merge,
    left_on=["enso_year", "season", "season_year"],
    right_on=["year", "season", "season_year"],
    how="left",
    validate="many_to_one",
    suffixes=("", "_enso")
)

if "year_enso" in pm25_onset_enso.columns:
    pm25_onset_enso = pm25_onset_enso.drop(columns=["year_enso"])

pm25_onset_enso["pollutant"] = "PM2.5"

print("PM10 onset table with ENSO:", pm10_onset_enso.shape)
print("PM2.5 onset table with ENSO:", pm25_onset_enso.shape)

print("\nMissing ENSO labels:")
print("PM10:", pm10_onset_enso["enso_phase"].isna().sum())
print("PM2.5:", pm25_onset_enso["enso_phase"].isna().sum())

print("\nDJF December check (PM10):")
display(
    pm10_onset_enso.loc[
        (pm10_onset_enso["season"] == "DJF") & (pm10_onset_enso["month"] == 12),
        ["day0", "year", "month", "season", "enso_year", "season_year", "enso_phase", "roni"]
    ].head(10)
)

print("\nDJF December check (PM2.5):")
display(
    pm25_onset_enso.loc[
        (pm25_onset_enso["season"] == "DJF") & (pm25_onset_enso["month"] == 12),
        ["day0", "year", "month", "season", "enso_year", "season_year", "enso_phase", "roni"]
    ].head(10)
)

PM10 onset table with ENSO: (250, 17)
PM2.5 onset table with ENSO: (286, 17)

Missing ENSO labels:
PM10: 0
PM2.5: 1

DJF December check (PM10):


,day0,year,month,season,enso_year,season_year,enso_phase,roni
37,2012-12-07,2012,12,DJF,2013,2013_DJF,La Niña,-0.63
38,2012-12-14,2012,12,DJF,2013,2013_DJF,La Niña,-0.63
39,2012-12-17,2012,12,DJF,2013,2013_DJF,La Niña,-0.63
40,2012-12-20,2012,12,DJF,2013,2013_DJF,La Niña,-0.63
41,2012-12-22,2012,12,DJF,2013,2013_DJF,La Niña,-0.63
42,2012-12-25,2012,12,DJF,2013,2013_DJF,La Niña,-0.63
43,2012-12-29,2012,12,DJF,2013,2013_DJF,La Niña,-0.63
75,2013-12-02,2013,12,DJF,2014,2014_DJF,Neutral,-0.49
76,2013-12-25,2013,12,DJF,2014,2014_DJF,Neutral,-0.49
99,2014-12-04,2014,12,DJF,2015,2015_DJF,Neutral,0.45



DJF December check (PM2.5):


,day0,year,month,season,enso_year,season_year,enso_phase,roni
27,2012-12-14,2012,12,DJF,2013,2013_DJF,La Niña,-0.63
28,2012-12-18,2012,12,DJF,2013,2013_DJF,La Niña,-0.63
29,2012-12-22,2012,12,DJF,2013,2013_DJF,La Niña,-0.63
30,2012-12-25,2012,12,DJF,2013,2013_DJF,La Niña,-0.63
66,2013-12-24,2013,12,DJF,2014,2014_DJF,Neutral,-0.49
94,2014-12-04,2014,12,DJF,2015,2015_DJF,Neutral,0.45
95,2014-12-31,2014,12,DJF,2015,2015_DJF,Neutral,0.45
129,2015-12-04,2015,12,DJF,2016,2016_DJF,El Niño,2.22
130,2015-12-10,2015,12,DJF,2016,2016_DJF,El Niño,2.22
131,2015-12-18,2015,12,DJF,2016,2016_DJF,El Niño,2.22


#### 10.2.2 Summarize onset counts and rates by ENSO phase

In [179]:
# Combine both pollutants
onset_enso_all = pd.concat(
    [pm10_onset_enso, pm25_onset_enso],
    ignore_index=True
)

# Number of available season-years in each season × ENSO phase
enso_phase_counts_clean = enso_phase_counts.copy()
enso_phase_counts_clean["season"] = pd.Categorical(
    enso_phase_counts_clean["season"],
    categories=SEASON_ORDER,
    ordered=True
)
enso_phase_counts_clean["enso_phase"] = pd.Categorical(
    enso_phase_counts_clean["enso_phase"],
    categories=ENSO_PHASE_ORDER,
    ordered=True
)

# Onset counts by pollutant × season × ENSO phase
onset_counts_by_phase = (
    onset_enso_all
    .groupby(["pollutant", "season", "enso_phase"], as_index=False)
    .size()
    .rename(columns={"size": "n_onset_events"})
)

onset_counts_by_phase["season"] = pd.Categorical(
    onset_counts_by_phase["season"],
    categories=SEASON_ORDER,
    ordered=True
)
onset_counts_by_phase["enso_phase"] = pd.Categorical(
    onset_counts_by_phase["enso_phase"],
    categories=ENSO_PHASE_ORDER,
    ordered=True
)

# Merge with number of season-years available
onset_counts_by_phase = onset_counts_by_phase.merge(
    enso_phase_counts_clean,
    on=["season", "enso_phase"],
    how="left",
    validate="many_to_one"
)

# Rate normalized by available season-years in each ENSO phase
onset_counts_by_phase["onset_events_per_season_year"] = (
    onset_counts_by_phase["n_onset_events"] / onset_counts_by_phase["n_season_years"]
)

# Within-season share by phase (optional but useful)
onset_counts_by_phase["season_total_onsets_for_pollutant"] = (
    onset_counts_by_phase
    .groupby(["pollutant", "season"])["n_onset_events"]
    .transform("sum")
)

onset_counts_by_phase["phase_share_within_season_pct"] = (
    100 * onset_counts_by_phase["n_onset_events"]
    / onset_counts_by_phase["season_total_onsets_for_pollutant"]
)

onset_counts_by_phase = onset_counts_by_phase.sort_values(
    ["pollutant", "season", "enso_phase"]
).reset_index(drop=True)

save_csv(onset_counts_by_phase, "stage5_onset_counts_rates_by_enso_phase.csv", index=False)

print("Saved: stage5_onset_counts_rates_by_enso_phase.csv")
onset_counts_by_phase

Saved: stage5_onset_counts_rates_by_enso_phase.csv


,pollutant,season,enso_phase,n_onset_events,n_season_years,onset_events_per_season_year,season_total_onsets_for_pollutant,phase_share_within_season_pct
0,PM10,DJF,El Niño,20,3,6.666667,70,28.571429
1,PM10,DJF,Neutral,13,3,4.333333,70,18.571429
2,PM10,DJF,La Niña,37,7,5.285714,70,52.857143
3,PM10,MAM,El Niño,9,3,3.000000,53,16.981132
4,PM10,MAM,Neutral,36,7,5.142857,53,67.924528
5,PM10,MAM,La Niña,8,3,2.666667,53,15.094340
6,PM10,JJA,El Niño,11,2,5.500000,68,16.176471
7,PM10,JJA,Neutral,41,6,6.833333,68,60.294118
8,PM10,JJA,La Niña,16,5,3.200000,68,23.529412
9,PM10,SON,El Niño,18,3,6.000000,59,30.508475


#### 10.2.3 Build compact comparison tables

In [180]:
# Pivot table: onset counts
onset_counts_pivot = onset_counts_by_phase.pivot_table(
    index=["pollutant", "season"],
    columns="enso_phase",
    values="n_onset_events"
).reset_index()

# Pivot table: onset rate per season-year
onset_rates_pivot = onset_counts_by_phase.pivot_table(
    index=["pollutant", "season"],
    columns="enso_phase",
    values="onset_events_per_season_year"
).reset_index()

# Flatten columns if needed
onset_counts_pivot.columns.name = None
onset_rates_pivot.columns.name = None

save_csv(onset_counts_pivot, "stage5_onset_counts_by_enso_phase_pivot.csv", index=False)
save_csv(onset_rates_pivot, "stage5_onset_rates_by_enso_phase_pivot.csv", index=False)

print("Saved: stage5_onset_counts_by_enso_phase_pivot.csv")
print("Saved: stage5_onset_rates_by_enso_phase_pivot.csv")

print("\nOnset counts by ENSO phase")
display(onset_counts_pivot)

print("\nOnset rates by ENSO phase (events per season-year)")
display(onset_rates_pivot)

Saved: stage5_onset_counts_by_enso_phase_pivot.csv
Saved: stage5_onset_rates_by_enso_phase_pivot.csv

Onset counts by ENSO phase


,pollutant,season,El Niño,Neutral,La Niña
0,PM10,DJF,20.0,13.0,37.0
1,PM10,MAM,9.0,36.0,8.0
2,PM10,JJA,11.0,41.0,16.0
3,PM10,SON,18.0,27.0,14.0
4,PM2.5,DJF,21.0,14.0,38.0
5,PM2.5,MAM,14.0,40.0,9.0
6,PM2.5,JJA,16.0,42.0,18.0
7,PM2.5,SON,19.0,27.0,27.0



Onset rates by ENSO phase (events per season-year)


,pollutant,season,El Niño,Neutral,La Niña
0,PM10,DJF,6.666667,4.333333,5.285714
1,PM10,MAM,3.000000,5.142857,2.666667
2,PM10,JJA,5.500000,6.833333,3.200000
3,PM10,SON,6.000000,6.750000,2.333333
4,PM2.5,DJF,7.000000,4.666667,5.428571
5,PM2.5,MAM,4.666667,5.714286,3.000000
6,PM2.5,JJA,8.000000,7.000000,3.600000
7,PM2.5,SON,6.333333,6.750000,4.500000


#### 10.2.4 Plot onset rates by ENSO phase

In [187]:
ENSO_COLORS = {
    "El Niño": "#D37259",   # warm
    "Neutral": "#C0BFBF",   # gray
    "La Niña": "#5DA5DA",   # cool
}

In [238]:
def plot_onset_rates_by_enso_phase(df_rates, pollutant_label, out_fig_path):
    """
    Grouped bar chart of onset rates by ENSO phase for one pollutant.

    y-axis:
        onset_events_per_season_year

    x-axis:
        DJF, MAM, JJA, SON

    colors:
        El Niño, Neutral, La Niña
    """
    sub = df_rates[df_rates["pollutant"] == pollutant_label].copy()

    # Keep plotting order
    sub["season"] = pd.Categorical(sub["season"], categories=SEASON_ORDER, ordered=True)
    sub["enso_phase"] = pd.Categorical(sub["enso_phase"], categories=ENSO_PHASE_ORDER, ordered=True)
    sub = sub.sort_values(["season", "enso_phase"]).reset_index(drop=True)

    # Pivot for grouped bars
    rates_pivot = sub.pivot(index="season", columns="enso_phase", values="onset_events_per_season_year")
    counts_pivot = sub.pivot(index="season", columns="enso_phase", values="n_onset_events")
    years_pivot = sub.pivot(index="season", columns="enso_phase", values="n_season_years")

    seasons = list(rates_pivot.index)
    x = np.arange(len(seasons))
    width = 0.22

    fig, ax = plt.subplots(figsize=(10.8, 6.2), dpi=250)

    for i, phase in enumerate(ENSO_PHASE_ORDER):
        offsets = x + (i - 1) * width
        values = rates_pivot[phase].values

        bars = ax.bar(
            offsets,
            values,
            width=width,
            label=phase,
            color=ENSO_COLORS[phase],
            edgecolor="black",
            linewidth=0.6
        )

        # Small value labels
        for bar, val in zip(bars, values):
            if np.isfinite(val):
                ax.text(
                    bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + 0.08,
                    f"{val:.1f}",
                    ha="center",
                    va="bottom",
                    fontsize=9
                )

    # Axes and grid
    ax.set_xticks(x)
    ax.set_xticklabels(seasons, fontsize=11)
    ax.set_ylabel("Onset events per season-year", fontsize=11)
    ax.grid(axis="y", linestyle="--", alpha=0.35)
    ax.set_axisbelow(True)

    # Set a little headroom for labels
    ymax = float(np.nanmax(rates_pivot.values))
    ax.set_ylim(0, ymax * 1.20)

    # Title block
    fig.suptitle(
        f"{pollutant_label}: seasonal onset frequency by ENSO phase",
        fontsize=17,
        fontweight="bold",
        y=0.98
    )
    fig.text(
        0.5, 0.91,
        "Bars show onset events per season-year using CPC RONI phase classification",
        ha="center",
        fontsize=11,
        style="italic"
    )
    fig.text(
        0.5, 0.875,
        "Rate = number of onset events divided by the number of available season-years in each ENSO phase",
        ha="center",
        fontsize=10
    )

    # Legend (outside plot)
    ax.legend(
        title="ENSO phase",
        loc="upper left",
        bbox_to_anchor=(0.9, -0.15),
        frameon=False
    )

    # Note with sample-year counts by season and phase
    note_lines = []
    for season in seasons:
        bits = []
        for phase in ENSO_PHASE_ORDER:
            n_events = int(counts_pivot.loc[season, phase])
            n_years = int(years_pivot.loc[season, phase])
            bits.append(f"{phase}: {n_events}/{n_years}y")
        note_lines.append(f"{season} → " + " | ".join(bits))

    fig.text(
        0.22, 0.04,
        "\n".join(note_lines),
        ha="center",
        fontsize=8
    )

    fig.subplots_adjust(top=0.78, bottom=0.24, left=0.09, right=0.96)

    plt.savefig(out_fig_path, dpi=300)
    plt.close(fig)
    print(f"Saved: {out_fig_path}")

In [239]:
plot_onset_rates_by_enso_phase(
    df_rates=onset_counts_by_phase,
    pollutant_label="PM10",
    out_fig_path=FIGURES_DIR / "stage5_PM10_onset_rates_by_enso_phase.png"
)

plot_onset_rates_by_enso_phase(
    df_rates=onset_counts_by_phase,
    pollutant_label="PM2.5",
    out_fig_path=FIGURES_DIR / "stage5_PM25_onset_rates_by_enso_phase.png"
)

Saved: C:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 5\outputs\figures\stage5_PM10_onset_rates_by_enso_phase.png
Saved: C:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 5\outputs\figures\stage5_PM25_onset_rates_by_enso_phase.png


### 10.3 Seasonal synoptic stories under ENSO

#### 10.3.1 Define ENSO-phase onset subsets and scope note

In [240]:
# Drop missing ENSO labels (there should be at most one late-DJF edge case)
pm10_onset_enso_valid = pm10_onset_enso.loc[pm10_onset_enso["enso_phase"].notna()].copy()
pm25_onset_enso_valid = pm25_onset_enso.loc[pm25_onset_enso["enso_phase"].notna()].copy()

# Scope note: which seasons are primary vs descriptive
enso_synoptic_scope = pd.DataFrame({
    "season": ["DJF", "MAM", "JJA", "SON"],
    "priority_level": [
        "Primary",
        "Primary",
        "Descriptive / conditional",
        "Descriptive / conditional"
    ],
    "reason": [
        "Winter benchmark and strongest teleconnection relevance",
        "Transition season highlighted in earlier stages",
        "Useful contrast, but some ENSO-phase bins have limited season-year support",
        "Useful context, but should not be overclaimed if visually weaker"
    ]
})

save_csv(enso_synoptic_scope, "stage5_enso_synoptic_scope_note.csv", index=False)

WindowsPath('C:/Users/DELL/OneDrive - TUNI.fi/Documents/Finlandia/Tampere Uni/Tesis/Schedule/Stage 5/outputs/tables/stage5_enso_synoptic_scope_note.csv')

In [241]:
# Build a support table of onset N and contributing season-years
def build_enso_synoptic_support_table(onset_df, pollutant_label):
    rows = []

    for season in SEASON_ORDER:
        for phase in ENSO_PHASE_ORDER:
            sub = onset_df.loc[
                (onset_df["season"] == season) &
                (onset_df["enso_phase"] == phase)
            ].copy()

            n_onset_days = len(sub)
            n_contributing_season_years = sub["season_year"].nunique()

            rows.append({
                "pollutant": pollutant_label,
                "season": season,
                "enso_phase": phase,
                "n_onset_days": n_onset_days,
                "n_contributing_season_years": n_contributing_season_years
            })

    return pd.DataFrame(rows)

In [242]:
pm10_enso_synoptic_support = build_enso_synoptic_support_table(pm10_onset_enso_valid, "PM10")
pm25_enso_synoptic_support = build_enso_synoptic_support_table(pm25_onset_enso_valid, "PM2.5")

enso_synoptic_support = pd.concat(
    [pm10_enso_synoptic_support, pm25_enso_synoptic_support],
    ignore_index=True
)

save_csv(enso_synoptic_support, "stage5_enso_synoptic_support_table.csv", index=False)

print("Saved: stage5_enso_synoptic_scope_note.csv")
print("Saved: stage5_enso_synoptic_support_table.csv")

display(enso_synoptic_scope)
display(enso_synoptic_support)

Saved: stage5_enso_synoptic_scope_note.csv
Saved: stage5_enso_synoptic_support_table.csv


,season,priority_level,reason
0,DJF,Primary,Winter benchmark and strongest teleconnection ...
1,MAM,Primary,Transition season highlighted in earlier stages
2,JJA,Descriptive / conditional,"Useful contrast, but some ENSO-phase bins have..."
3,SON,Descriptive / conditional,"Useful context, but should not be overclaimed ..."


,pollutant,season,enso_phase,n_onset_days,n_contributing_season_years
0,PM10,DJF,El Niño,20,3
1,PM10,DJF,Neutral,13,3
2,PM10,DJF,La Niña,37,7
3,PM10,MAM,El Niño,9,3
4,PM10,MAM,Neutral,36,6
5,PM10,MAM,La Niña,8,3
6,PM10,JJA,El Niño,11,1
7,PM10,JJA,Neutral,41,6
8,PM10,JJA,La Niña,16,5
9,PM10,SON,El Niño,18,2


#### 10.3.2 Plot seasonal day-0 composites by ENSO phase

In [249]:
def safe_composite_from_dates(da: xr.DataArray, dates: pd.DatetimeIndex) -> xr.DataArray:
    """
    Compute mean composite over selected dates.
    If dates are empty, return a NaN field with the same y/x shape.
    """
    if len(dates) == 0:
        return da.isel(time=0).copy() * np.nan
    return da.sel(time=dates).mean("time")

def build_enso_phase_composite_dict(onset_df, z_anom, u_anom, v_anom, season):
    """
    Build a 3-phase composite dictionary for a single season:
    El Niño, Neutral, La Niña
    """
    out = {}

    for phase in ENSO_PHASE_ORDER:
        sub = onset_df.loc[
            (onset_df["season"] == season) &
            (onset_df["enso_phase"] == phase)
        ].copy()

        dates = pd.DatetimeIndex(sorted(pd.to_datetime(sub["day0"]).unique()))
        dates = keep_only_available_dates(dates, available_time_index)

        out[phase] = {
            "dates": dates,
            "n_onset_days": len(dates),
            "n_contributing_season_years": sub["season_year"].nunique(),
            "z": safe_composite_from_dates(z_anom, dates),
            "u": safe_composite_from_dates(u_anom, dates),
            "v": safe_composite_from_dates(v_anom, dates),
        }

    return out

def plot_enso_phase_composite_panel(
    comp_dict,
    pollutant_label,
    season,
    out_fig_path,
    vector_step=6,
    contour_step=5
):
    """
    3-panel figure: El Niño, Neutral, La Niña
    for one pollutant and one season.
    """
    import matplotlib.patches as mpatches

    max_abs = max(
        float(np.nanmax(np.abs(comp_dict[phase]["z"].values)))
        if np.isfinite(np.nanmax(np.abs(comp_dict[phase]["z"].values))) else 0.0
        for phase in ENSO_PHASE_ORDER
    )
    if not np.isfinite(max_abs) or max_abs == 0:
        max_abs = 1.0

    norm = TwoSlopeNorm(vcenter=0, vmin=-max_abs, vmax=max_abs)

    proj = ccrs.PlateCarree()
    fig, axes = plt.subplots(
        1, 3,
        figsize=(16.2, 5.8),
        dpi=250,
        subplot_kw={"projection": proj}
    )

    fig.subplots_adjust(
        left=0.06,
        right=0.95,
        bottom=0.18,
        top=0.80,
        wspace=0.10
    )

    pcm_ref = None

    for i, phase in enumerate(ENSO_PHASE_ORDER):
        ax = axes[i]

        z = comp_dict[phase]["z"].values
        u = comp_dict[phase]["u"].values
        v = comp_dict[phase]["v"].values
        n_used = comp_dict[phase]["n_onset_days"]
        n_years = comp_dict[phase]["n_contributing_season_years"]

        ax.set_extent(
            [
                DOMAIN_MAIN["lon_min"], DOMAIN_MAIN["lon_max"],
                DOMAIN_MAIN["lat_min"], DOMAIN_MAIN["lat_max"]
            ],
            crs=proj
        )

        ax.coastlines(resolution="50m", linewidth=0.5)
        ax.add_feature(cfeature.BORDERS, linewidth=0.4)
        ax.add_feature(cfeature.STATES.with_scale("50m"), linewidth=0.3)

        # Shading
        pcm = ax.pcolormesh(
            lon2d, lat2d, z,
            cmap="RdBu_r",
            norm=norm,
            shading="auto",
            transform=proj
        )
        pcm_ref = pcm

        # Contours
        lev = np.arange(-max_abs, max_abs + contour_step, contour_step)
        pos = lev[lev > 0]
        neg = lev[lev < 0]

        if len(pos) > 0:
            ax.contour(
                lon2d, lat2d, z,
                levels=pos,
                colors="k",
                linewidths=0.7,
                transform=proj
            )
        if len(neg) > 0:
            ax.contour(
                lon2d, lat2d, z,
                levels=neg,
                colors="k",
                linewidths=0.7,
                linestyles="--",
                transform=proj
            )

        # Vectors
        yy = np.arange(0, z.shape[0], vector_step)
        xx = np.arange(0, z.shape[1], vector_step)

        ax.quiver(
            lon2d[np.ix_(yy, xx)],
            lat2d[np.ix_(yy, xx)],
            u[np.ix_(yy, xx)],
            v[np.ix_(yy, xx)],
            transform=proj,
            color="black",
            scale=80,
            width=0.0022,
            headwidth=3.5,
            headlength=4.5,
            headaxislength=4.0,
            minlength=0.05,
            pivot="middle",
            zorder=4
        )

        # Valley of Mexico box
        rect = mpatches.Rectangle(
            (VOM_BOX[0], VOM_BOX[1]),
            VOM_BOX[2], VOM_BOX[3],
            fill=False,
            edgecolor="k",
            linewidth=1.0,
            linestyle="-",
            transform=proj
        )
        ax.add_patch(rect)

        # CDMX star
        ax.plot(
            CDMX_REF["lon"], CDMX_REF["lat"],
            marker="*",
            color="gold",
            markersize=9,
            markeredgecolor="k",
            transform=proj
        )

        ax.set_title(
            f"{phase} (N={n_used}, y={n_years})",
            fontsize=13,
            weight="bold"
        )

        gl = ax.gridlines(
            draw_labels=True,
            linewidth=0.2,
            color="gray",
            alpha=0.5,
            linestyle="--"
        )
        gl.top_labels = False
        gl.right_labels = False
        gl.bottom_labels = True
        gl.left_labels = (i == 0)

    # Colorbar
    cbar_ax = fig.add_axes([0.32, 0.1, 0.36, 0.055])
    cb = fig.colorbar(pcm_ref, cax=cbar_ax, orientation="horizontal", extend="both")
    cb.set_label("Z500′ anomaly (m)", fontsize=12)
    cb.ax.tick_params(labelsize=10)

    # Titles
    fig.suptitle(
        f"{pollutant_label}: {season} onset-day synoptic story by ENSO phase (2012–2024)",
        fontsize=20,
        weight="bold",
        y=0.975
    )
    fig.text(
        0.5, 0.895,
        "Shading/contours: Z500′ anomaly | Vectors: Δ(U′,V′) at 500 hPa | Day 0 = onset",
        ha="center",
        fontsize=15,
        style="italic"
    )
    fig.text(
        0.5, 0.86,
        "Panels compare CPC RONI phase classes within a fixed season | N = onset days, y = contributing season-years",
        ha="center",
        fontsize=12
    )

    # Legend
    legend_elements = [
        mpatches.Rectangle((0, 0), 1, 1, fill=False, edgecolor="k", linewidth=1.5, label="Valley of Mexico"),
        plt.Line2D(
            [0], [0],
            marker="*",
            color="w",
            markerfacecolor="gold",
            markeredgecolor="k",
            markersize=12,
            label="Mexico City (CDMX)"
        )
    ]
    fig.legend(
        handles=legend_elements,
        loc="upper left",
        fontsize=11,
        frameon=False,
        bbox_to_anchor=(0.03, 0.89)
    )

    plt.savefig(out_fig_path, dpi=300)
    plt.close(fig)
    print(f"Saved: {out_fig_path}")

In [250]:
# PM10
for season in SEASON_ORDER:
    comp_dict = build_enso_phase_composite_dict(
        onset_df=pm10_onset_enso_valid,
        z_anom=z500_anom,
        u_anom=u500_anom,
        v_anom=v500_anom,
        season=season
    )

    out_fig = FIGURES_DIR / f"stage5_PM10_{season}_enso_phase_day0_3panel.png"
    plot_enso_phase_composite_panel(
        comp_dict=comp_dict,
        pollutant_label="PM10",
        season=season,
        out_fig_path=out_fig,
        vector_step=6,
        contour_step=5
    )

Saved: C:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 5\outputs\figures\stage5_PM10_DJF_enso_phase_day0_3panel.png
Saved: C:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 5\outputs\figures\stage5_PM10_MAM_enso_phase_day0_3panel.png
Saved: C:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 5\outputs\figures\stage5_PM10_JJA_enso_phase_day0_3panel.png
Saved: C:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 5\outputs\figures\stage5_PM10_SON_enso_phase_day0_3panel.png


In [251]:
# PM2.5
for season in SEASON_ORDER:
    comp_dict = build_enso_phase_composite_dict(
        onset_df=pm25_onset_enso_valid,
        z_anom=z500_anom,
        u_anom=u500_anom,
        v_anom=v500_anom,
        season=season
    )

    out_fig = FIGURES_DIR / f"stage5_PM25_{season}_enso_phase_day0_3panel.png"
    plot_enso_phase_composite_panel(
        comp_dict=comp_dict,
        pollutant_label="PM2.5",
        season=season,
        out_fig_path=out_fig,
        vector_step=6,
        contour_step=5
    )

Saved: C:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 5\outputs\figures\stage5_PM25_DJF_enso_phase_day0_3panel.png
Saved: C:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 5\outputs\figures\stage5_PM25_MAM_enso_phase_day0_3panel.png
Saved: C:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 5\outputs\figures\stage5_PM25_JJA_enso_phase_day0_3panel.png
Saved: C:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 5\outputs\figures\stage5_PM25_SON_enso_phase_day0_3panel.png


### 10.4 WHO exceedance diagnostics by ENSO phase

#### 10.4.1 Attach ENSO phase to daily PM series

In [252]:
WHO_PM10 = 45.0
WHO_PM25 = 15.0

pm_daily_enso = pm_daily.copy()

# Standard date parts
pm_daily_enso["year"] = pm_daily_enso["date"].dt.year
pm_daily_enso["month"] = pm_daily_enso["date"].dt.month
pm_daily_enso["season"] = pm_daily_enso["month"].map(MONTH_TO_SEASON)

# ENSO-year logic for DJF:
# December belongs to the following DJF season-year
pm_daily_enso["enso_year"] = pm_daily_enso["year"]
pm_daily_enso.loc[
    (pm_daily_enso["season"] == "DJF") & (pm_daily_enso["month"] == 12),
    "enso_year"
] = (
    pm_daily_enso.loc[
        (pm_daily_enso["season"] == "DJF") & (pm_daily_enso["month"] == 12),
        "enso_year"
    ] + 1
)

pm_daily_enso["season_year"] = (
    pm_daily_enso["enso_year"].astype(str) + "_" + pm_daily_enso["season"].astype(str)
)

enso_merge = enso_lookup[["year", "season", "season_year", "roni", "enso_phase", "enso_phase_short"]].copy()

pm_daily_enso = pm_daily_enso.merge(
    enso_merge,
    left_on=["enso_year", "season", "season_year"],
    right_on=["year", "season", "season_year"],
    how="left",
    validate="many_to_one",
    suffixes=("", "_enso")
)

if "year_enso" in pm_daily_enso.columns:
    pm_daily_enso = pm_daily_enso.drop(columns=["year_enso"])

# WHO exceedance diagnostics
pm_daily_enso["PM10_excess"] = (pm_daily_enso["PM10"] - WHO_PM10).clip(lower=0)
pm_daily_enso["PM25_excess"] = (pm_daily_enso["PM2.5"] - WHO_PM25).clip(lower=0)

pm_daily_enso["PM10_exceed"] = pm_daily_enso["PM10_excess"] > 0
pm_daily_enso["PM25_exceed"] = pm_daily_enso["PM25_excess"] > 0

print("PM daily series with ENSO labels:", pm_daily_enso.shape)
print("\nMissing ENSO labels:")
print(pm_daily_enso["enso_phase"].isna().sum())

pm_daily_enso.head()

PM daily series with ENSO labels: (4555, 15)

Missing ENSO labels:
31


,date,PM10,PM2.5,year,month,season,enso_year,season_year,roni,enso_phase,enso_phase_short,PM10_excess,PM25_excess,PM10_exceed,PM25_exceed
0,2012-01-01,100.14,66.43,2012,1,DJF,2012,2012_DJF,-0.82,La Niña,LN,55.14,51.43,True,True
1,2012-01-02,19.29,6.14,2012,1,DJF,2012,2012_DJF,-0.82,La Niña,LN,0.00,0.00,False,False
2,2012-01-03,38.00,17.43,2012,1,DJF,2012,2012_DJF,-0.82,La Niña,LN,0.00,2.43,False,True
3,2012-01-04,67.71,35.00,2012,1,DJF,2012,2012_DJF,-0.82,La Niña,LN,22.71,20.00,True,True
4,2012-01-05,61.43,28.86,2012,1,DJF,2012,2012_DJF,-0.82,La Niña,LN,16.43,13.86,True,True


#### 10.4.2 Compute seasonal WHO exceedance diagnostics by ENSO phase

In [253]:
def compute_who_diag_for_group(df_group, pollutant_col, excess_col, exceed_col):
    """
    Compute WHO exceedance diagnostics for one subset:
    - exceedance frequency (% of days)
    - mean excess magnitude on exceedance days
    - total excess load (sum of exceedance amount)
    """
    n_days = len(df_group)
    n_exceed = int(df_group[exceed_col].sum())

    exceed_pct = 100.0 * n_exceed / n_days if n_days > 0 else np.nan
    excess_mag_mean = (
        df_group.loc[df_group[exceed_col], excess_col].mean()
        if n_exceed > 0 else 0.0
    )
    excess_load_sum = df_group[excess_col].sum()

    return {
        "n_days": n_days,
        "n_exceed_days": n_exceed,
        "exceed_pct": exceed_pct,
        "excess_mag_mean": float(excess_mag_mean),
        "excess_load_sum": float(excess_load_sum),
    }

In [254]:
rows = []

for season in SEASON_ORDER:
    for phase in ENSO_PHASE_ORDER:
        sub = pm_daily_enso.loc[
            (pm_daily_enso["season"] == season) &
            (pm_daily_enso["enso_phase"] == phase)
        ].copy()

        if len(sub) == 0:
            continue

        out10 = compute_who_diag_for_group(sub, "PM10", "PM10_excess", "PM10_exceed")
        out25 = compute_who_diag_for_group(sub, "PM2.5", "PM25_excess", "PM25_exceed")

        rows.append({
            "season": season,
            "enso_phase": phase,

            "n_days": len(sub),
            "n_season_years": sub["season_year"].nunique(),

            "PM10_exceed_pct": out10["exceed_pct"],
            "PM10_excess_mag_mean": out10["excess_mag_mean"],
            "PM10_excess_load_sum": out10["excess_load_sum"],
            "PM10_n_exceed_days": out10["n_exceed_days"],

            "PM25_exceed_pct": out25["exceed_pct"],
            "PM25_excess_mag_mean": out25["excess_mag_mean"],
            "PM25_excess_load_sum": out25["excess_load_sum"],
            "PM25_n_exceed_days": out25["n_exceed_days"],
        })

who_enso_seasonal = pd.DataFrame(rows)

who_enso_seasonal["season"] = pd.Categorical(
    who_enso_seasonal["season"],
    categories=SEASON_ORDER,
    ordered=True
)
who_enso_seasonal["enso_phase"] = pd.Categorical(
    who_enso_seasonal["enso_phase"],
    categories=ENSO_PHASE_ORDER,
    ordered=True
)

who_enso_seasonal = who_enso_seasonal.sort_values(
    ["season", "enso_phase"]
).reset_index(drop=True)

save_csv(who_enso_seasonal, "stage5_who_exceedance_by_enso_phase_seasonal.csv", index=False)

print("Saved: stage5_who_exceedance_by_enso_phase_seasonal.csv")
who_enso_seasonal

Saved: stage5_who_exceedance_by_enso_phase_seasonal.csv


,season,enso_phase,n_days,n_season_years,PM10_exceed_pct,PM10_excess_mag_mean,PM10_excess_load_sum,PM10_n_exceed_days,PM25_exceed_pct,PM25_excess_mag_mean,PM25_excess_load_sum,PM25_n_exceed_days
0,DJF,El Niño,272,3,65.073529,20.414802,3613.42,177,84.191176,15.350961,3515.37,229
1,DJF,Neutral,271,3,72.693727,18.544365,3653.24,197,88.929889,14.234979,3430.63,241
2,DJF,La Niña,552,7,79.528986,19.030159,8354.24,439,92.572464,13.696399,6998.86,511
3,MAM,El Niño,276,3,65.942029,13.434231,2445.03,182,95.289855,13.514563,3554.33,263
4,MAM,Neutral,644,7,67.701863,15.962271,6959.55,436,94.720497,12.483443,7614.90,610
5,MAM,La Niña,234,3,61.111111,12.471119,1783.37,143,93.589744,10.600000,2321.40,219
6,JJA,El Niño,184,2,14.673913,6.705185,181.04,27,67.934783,7.440960,930.12,125
7,JJA,Neutral,552,6,19.021739,6.378381,669.73,105,78.079710,6.455313,2782.24,431
8,JJA,La Niña,418,5,7.416268,6.130323,190.04,31,60.287081,5.760198,1451.57,252
9,SON,El Niño,273,3,24.542125,13.165373,882.08,67,74.725275,8.815147,1798.29,204


#### 10.4.3 Compute pooled WHO exceedance diagnostics by ENSO phase

In [255]:
rows = []

for phase in ENSO_PHASE_ORDER:
    sub = pm_daily_enso.loc[pm_daily_enso["enso_phase"] == phase].copy()

    if len(sub) == 0:
        continue

    out10 = compute_who_diag_for_group(sub, "PM10", "PM10_excess", "PM10_exceed")
    out25 = compute_who_diag_for_group(sub, "PM2.5", "PM25_excess", "PM25_exceed")

    rows.append({
        "enso_phase": phase,
        "n_days": len(sub),
        "n_unique_season_years": sub["season_year"].nunique(),

        "PM10_exceed_pct": out10["exceed_pct"],
        "PM10_excess_mag_mean": out10["excess_mag_mean"],
        "PM10_excess_load_sum": out10["excess_load_sum"],
        "PM10_n_exceed_days": out10["n_exceed_days"],

        "PM25_exceed_pct": out25["exceed_pct"],
        "PM25_excess_mag_mean": out25["excess_mag_mean"],
        "PM25_excess_load_sum": out25["excess_load_sum"],
        "PM25_n_exceed_days": out25["n_exceed_days"],
    })

who_enso_pooled = pd.DataFrame(rows)

who_enso_pooled["enso_phase"] = pd.Categorical(
    who_enso_pooled["enso_phase"],
    categories=ENSO_PHASE_ORDER,
    ordered=True
)

who_enso_pooled = who_enso_pooled.sort_values("enso_phase").reset_index(drop=True)

save_csv(who_enso_pooled, "stage5_who_exceedance_by_enso_phase_pooled.csv", index=False)

print("Saved: stage5_who_exceedance_by_enso_phase_pooled.csv")
who_enso_pooled

Saved: stage5_who_exceedance_by_enso_phase_pooled.csv


,enso_phase,n_days,n_unique_season_years,PM10_exceed_pct,PM10_excess_mag_mean,PM10_excess_load_sum,PM10_n_exceed_days,PM25_exceed_pct,PM25_excess_mag_mean,PM25_excess_load_sum,PM25_n_exceed_days
0,El Niño,1005,11,45.074627,15.720905,7121.57,453,81.691542,11.934361,9798.11,821
1,Neutral,1831,20,47.078099,14.923956,12864.45,862,84.434735,10.583312,16361.80,1546
2,La Niña,1688,21,43.483412,15.814223,11607.64,734,76.895735,10.261941,13320.00,1298


#### 10.4.4 Comparison tables

In [257]:
# PM10 seasonal pivots
pm10_who_pct_pivot = who_enso_seasonal.pivot_table(
    index="season",
    columns="enso_phase",
    values="PM10_exceed_pct"
).reset_index()

pm10_who_load_pivot = who_enso_seasonal.pivot_table(
    index="season",
    columns="enso_phase",
    values="PM10_excess_load_sum"
).reset_index()

# PM2.5 seasonal pivots
pm25_who_pct_pivot = who_enso_seasonal.pivot_table(
    index="season",
    columns="enso_phase",
    values="PM25_exceed_pct"
).reset_index()

pm25_who_load_pivot = who_enso_seasonal.pivot_table(
    index="season",
    columns="enso_phase",
    values="PM25_excess_load_sum"
).reset_index()

for df in [pm10_who_pct_pivot, pm10_who_load_pivot, pm25_who_pct_pivot, pm25_who_load_pivot]:
    df.columns.name = None

save_csv(pm10_who_pct_pivot, "stage5_PM10_who_exceed_pct_by_enso_phase_pivot.csv", index=False)
save_csv(pm10_who_load_pivot, "stage5_PM10_who_excess_load_by_enso_phase_pivot.csv", index=False)
save_csv(pm25_who_pct_pivot, "stage5_PM25_who_exceed_pct_by_enso_phase_pivot.csv", index=False)
save_csv(pm25_who_load_pivot, "stage5_PM25_who_excess_load_by_enso_phase_pivot.csv", index=False)

print("Saved: stage5_PM10_who_exceed_pct_by_enso_phase_pivot.csv")
print("Saved: stage5_PM10_who_excess_load_by_enso_phase_pivot.csv")
print("Saved: stage5_PM25_who_exceed_pct_by_enso_phase_pivot.csv")
print("Saved: stage5_PM25_who_excess_load_by_enso_phase_pivot.csv")

print("\nPM10 WHO exceedance by ENSO phase")
display(pm10_who_pct_pivot)

print("\nPM10 WHO excess load by ENSO phase")
display(pm10_who_load_pivot)

print("\nPM2.5 WHO exceedance by ENSO phase")
display(pm25_who_pct_pivot)

print("\nPM2.5 WHO excess load by ENSO phase")
display(pm25_who_load_pivot)

Saved: stage5_PM10_who_exceed_pct_by_enso_phase_pivot.csv
Saved: stage5_PM10_who_excess_load_by_enso_phase_pivot.csv
Saved: stage5_PM25_who_exceed_pct_by_enso_phase_pivot.csv
Saved: stage5_PM25_who_excess_load_by_enso_phase_pivot.csv

PM10 WHO exceedance by ENSO phase


,season,El Niño,Neutral,La Niña
0,DJF,65.073529,72.693727,79.528986
1,MAM,65.942029,67.701863,61.111111
2,JJA,14.673913,19.021739,7.416268
3,SON,24.542125,34.065934,25.000000



PM10 WHO excess load by ENSO phase


,season,El Niño,Neutral,La Niña
0,DJF,3613.42,3653.24,8354.24
1,MAM,2445.03,6959.55,1783.37
2,JJA,181.04,669.73,190.04
3,SON,882.08,1581.93,1279.99



PM2.5 WHO exceedance by ENSO phase


,season,El Niño,Neutral,La Niña
0,DJF,84.191176,88.929889,92.572464
1,MAM,95.289855,94.720497,93.589744
2,JJA,67.934783,78.079710,60.287081
3,SON,74.725275,72.527473,65.289256



PM2.5 WHO excess load by ENSO phase


,season,El Niño,Neutral,La Niña
0,DJF,3515.37,3430.63,6998.86
1,MAM,3554.33,7614.90,2321.40
2,JJA,930.12,2782.24,1451.57
3,SON,1798.29,2534.03,2548.17


### 10.5 Robustness checks

#### 10.5.1 Statistical scope and test design

In [259]:
robustness_scope_note = pd.DataFrame({
    "component": [
        "Onset frequency by ENSO phase",
        "WHO exceedance frequency by ENSO phase",
        "WHO excess magnitude",
        "WHO excess load"
    ],
    "treatment": [
        "Formal chi-square robustness check",
        "Formal chi-square robustness check",
        "Descriptive only",
        "Descriptive only"
    ],
    "reason": [
        "Counts can be compared against phase availability (season-years).",
        "Exceed / non-exceed counts can be tested across ENSO phases.",
        "Magnitude distributions are not the primary robustness target in this thesis stage.",
        "Accumulated load depends strongly on sample length and is better treated descriptively."
    ]
})

save_csv(robustness_scope_note, "stage5_enso_robustness_scope_note.csv", index=False)

print("Saved: stage5_enso_robustness_scope_note.csv")
robustness_scope_note

Saved: stage5_enso_robustness_scope_note.csv


,component,treatment,reason
0,Onset frequency by ENSO phase,Formal chi-square robustness check,Counts can be compared against phase availabil...
1,WHO exceedance frequency by ENSO phase,Formal chi-square robustness check,Exceed / non-exceed counts can be tested acros...
2,WHO excess magnitude,Descriptive only,Magnitude distributions are not the primary ro...
3,WHO excess load,Descriptive only,Accumulated load depends strongly on sample le...


#### 10.5.2 Onset frequency robustness by ENSO phase

In [260]:
def onset_phase_chisquare_test(df_counts, pollutant, season):
    """
    Chi-square goodness-of-fit test for onset counts by ENSO phase,
    using expected counts proportional to the number of available season-years.
    """
    sub = df_counts.loc[
        (df_counts["pollutant"] == pollutant) &
        (df_counts["season"] == season)
    ].copy()

    sub["enso_phase"] = pd.Categorical(sub["enso_phase"], categories=ENSO_PHASE_ORDER, ordered=True)
    sub = sub.sort_values("enso_phase").reset_index(drop=True)

    obs = sub["n_onset_events"].values.astype(float)
    years = sub["n_season_years"].values.astype(float)

    total_obs = obs.sum()
    expected = total_obs * (years / years.sum())

    chi2_stat, pval = chisquare(f_obs=obs, f_exp=expected)

    return {
        "pollutant": pollutant,
        "season": season,
        "obs_el_nino": obs[0],
        "obs_neutral": obs[1],
        "obs_la_nina": obs[2],
        "exp_el_nino": expected[0],
        "exp_neutral": expected[1],
        "exp_la_nina": expected[2],
        "chi2_stat": float(chi2_stat),
        "p_value": float(pval),
        "min_expected_count": float(expected.min()),
        "result_note": (
            "Phase-dependent onset frequency suggested"
            if pval < 0.05 else
            "No strong evidence against phase-proportional onset frequency"
        )
    }

In [262]:
rows = []
for pollutant in ["PM10", "PM2.5"]:
    for season in SEASON_ORDER:
        rows.append(onset_phase_chisquare_test(onset_counts_by_phase, pollutant, season))

onset_enso_robustness = pd.DataFrame(rows)

# Add a significance flag
onset_enso_robustness["significant_at_0p05"] = onset_enso_robustness["p_value"] < 0.05

save_csv(onset_enso_robustness, "stage5_onset_frequency_enso_chisquare.csv", index=False)

print("Saved: stage5_onset_frequency_enso_chisquare.csv")
onset_enso_robustness

Saved: stage5_onset_frequency_enso_chisquare.csv


,pollutant,season,obs_el_nino,obs_neutral,obs_la_nina,exp_el_nino,exp_neutral,exp_la_nina,chi2_stat,p_value,min_expected_count,result_note,significant_at_0p05
0,PM10,DJF,20.0,13.0,37.0,16.153846,16.153846,37.692308,1.544218,0.462038,16.153846,No strong evidence against phase-proportional ...,False
1,PM10,MAM,9.0,36.0,8.0,12.230769,28.538462,12.230769,4.267745,0.118378,12.230769,No strong evidence against phase-proportional ...,False
2,PM10,JJA,11.0,41.0,16.0,10.461538,31.384615,26.153846,6.915686,0.031498,10.461538,Phase-dependent onset frequency suggested,True
3,PM10,SON,18.0,27.0,14.0,13.615385,18.153846,27.230769,12.151130,0.002298,13.615385,Phase-dependent onset frequency suggested,True
4,PM2.5,DJF,21.0,14.0,38.0,16.846154,16.846154,39.307692,1.548598,0.461027,16.846154,No strong evidence against phase-proportional ...,False
5,PM2.5,MAM,14.0,40.0,9.0,14.538462,33.923077,14.538462,3.218443,0.200043,14.538462,No strong evidence against phase-proportional ...,False
6,PM2.5,JJA,16.0,42.0,18.0,11.692308,35.076923,29.230769,7.268421,0.026405,11.692308,Phase-dependent onset frequency suggested,True
7,PM2.5,SON,19.0,27.0,27.0,16.846154,22.461538,33.692308,2.521689,0.283415,16.846154,No strong evidence against phase-proportional ...,False


#### 10.5.3 WHO exceedance frequency robustness by ENSO phase

In [263]:
def cramers_v_from_contingency(table):
    """
    Cramer's V for a contingency table.
    """
    chi2_stat, _, _, _ = chi2_contingency(table)
    n = table.sum()
    r, k = table.shape
    return np.sqrt(chi2_stat / (n * (min(r, k) - 1)))

def who_exceedance_chisquare_test(df_who_seasonal, pollutant, season):
    """
    Chi-square test of independence for exceed vs non-exceed across ENSO phases.
    Uses a 2 x 3 table:
        rows   = exceed / non-exceed
        cols   = El Niño / Neutral / La Niña
    """
    sub = df_who_seasonal.loc[df_who_seasonal["season"] == season].copy()
    sub["enso_phase"] = pd.Categorical(sub["enso_phase"], categories=ENSO_PHASE_ORDER, ordered=True)
    sub = sub.sort_values("enso_phase").reset_index(drop=True)

    if pollutant == "PM10":
        exceed = sub["PM10_n_exceed_days"].values.astype(int)
    else:
        exceed = sub["PM25_n_exceed_days"].values.astype(int)

    total_days = sub["n_days"].values.astype(int)
    non_exceed = total_days - exceed

    contingency = np.vstack([exceed, non_exceed])

    chi2_stat, pval, dof, expected = chi2_contingency(contingency)
    cramers_v = cramers_v_from_contingency(contingency)

    return {
        "pollutant": pollutant,
        "season": season,
        "exceed_el_nino": int(exceed[0]),
        "exceed_neutral": int(exceed[1]),
        "exceed_la_nina": int(exceed[2]),
        "nonexceed_el_nino": int(non_exceed[0]),
        "nonexceed_neutral": int(non_exceed[1]),
        "nonexceed_la_nina": int(non_exceed[2]),
        "chi2_stat": float(chi2_stat),
        "p_value": float(pval),
        "dof": int(dof),
        "min_expected_cell": float(expected.min()),
        "cramers_v": float(cramers_v),
        "result_note": (
            "Exceedance frequency differs by ENSO phase"
            if pval < 0.05 else
            "No strong evidence that exceedance frequency differs by ENSO phase"
        )
    }

In [264]:
rows = []
for pollutant in ["PM10", "PM2.5"]:
    for season in SEASON_ORDER:
        rows.append(who_exceedance_chisquare_test(who_enso_seasonal, pollutant, season))

who_enso_robustness = pd.DataFrame(rows)
who_enso_robustness["significant_at_0p05"] = who_enso_robustness["p_value"] < 0.05

save_csv(who_enso_robustness, "stage5_who_exceedance_enso_chisquare.csv", index=False)

print("Saved: stage5_who_exceedance_enso_chisquare.csv")
who_enso_robustness

Saved: stage5_who_exceedance_enso_chisquare.csv


,pollutant,season,exceed_el_nino,exceed_neutral,exceed_la_nina,nonexceed_el_nino,nonexceed_neutral,nonexceed_la_nina,chi2_stat,p_value,dof,min_expected_cell,cramers_v,result_note,significant_at_0p05
0,PM10,DJF,177,197,439,95,74,113,20.366982,3.778905e-05,2,69.791781,0.136382,Exceedance frequency differs by ENSO phase,True
1,PM10,MAM,182,436,143,94,208,91,3.319799,1.901581e-01,2,79.689775,0.053636,No strong evidence that exceedance frequency d...,False
2,PM10,JJA,27,105,31,157,447,387,26.467529,1.789160e-06,2,25.989601,0.151445,Exceedance frequency differs by ENSO phase,True
3,PM10,SON,67,124,121,206,240,363,10.446263,5.390423e-03,2,75.982159,0.096533,Exceedance frequency differs by ENSO phase,True
4,PM2.5,DJF,229,241,511,43,30,41,13.890996,9.629608e-04,2,28.213699,0.112631,Exceedance frequency differs by ENSO phase,True
5,PM2.5,MAM,263,610,219,13,34,15,0.744809,6.890756e-01,2,12.571924,0.025405,No strong evidence that exceedance frequency d...,False
6,PM2.5,JJA,125,431,252,59,121,166,36.323612,1.295472e-08,2,55.168111,0.177416,Exceedance frequency differs by ENSO phase,True
7,PM2.5,SON,204,264,316,69,100,168,9.111562,1.050629e-02,2,82.070473,0.090156,Exceedance frequency differs by ENSO phase,True


#### 10.5.4 Descriptive burden support table

In [265]:
rows = []

for season in SEASON_ORDER:
    sub = who_enso_seasonal.loc[who_enso_seasonal["season"] == season].copy()
    sub["enso_phase"] = pd.Categorical(sub["enso_phase"], categories=ENSO_PHASE_ORDER, ordered=True)
    sub = sub.sort_values("enso_phase").reset_index(drop=True)

    # PM10
    pm10_rank_pct = sub[["enso_phase", "PM10_exceed_pct"]].sort_values("PM10_exceed_pct", ascending=False)
    pm10_rank_load = sub[["enso_phase", "PM10_excess_load_sum"]].sort_values("PM10_excess_load_sum", ascending=False)

    # PM2.5
    pm25_rank_pct = sub[["enso_phase", "PM25_exceed_pct"]].sort_values("PM25_exceed_pct", ascending=False)
    pm25_rank_load = sub[["enso_phase", "PM25_excess_load_sum"]].sort_values("PM25_excess_load_sum", ascending=False)

    rows.append({
        "season": season,
        "PM10_top_phase_exceed_pct": pm10_rank_pct.iloc[0]["enso_phase"],
        "PM10_top_phase_excess_load": pm10_rank_load.iloc[0]["enso_phase"],
        "PM25_top_phase_exceed_pct": pm25_rank_pct.iloc[0]["enso_phase"],
        "PM25_top_phase_excess_load": pm25_rank_load.iloc[0]["enso_phase"],
    })

enso_burden_support = pd.DataFrame(rows)

save_csv(enso_burden_support, "stage5_enso_burden_support_table.csv", index=False)

print("Saved: stage5_enso_burden_support_table.csv")
enso_burden_support

Saved: stage5_enso_burden_support_table.csv


,season,PM10_top_phase_exceed_pct,PM10_top_phase_excess_load,PM25_top_phase_exceed_pct,PM25_top_phase_excess_load
0,DJF,La Niña,La Niña,La Niña,La Niña
1,MAM,Neutral,Neutral,El Niño,Neutral
2,JJA,Neutral,Neutral,Neutral,Neutral
3,SON,Neutral,Neutral,El Niño,La Niña


### 10.6 Interpretation notes and final ENSO synthesis

In [266]:
enso_final_synthesis = pd.DataFrame({
    "theme": [
        "Overall role of ENSO",
        "DJF",
        "MAM",
        "JJA",
        "SON",
        "Frequency vs burden",
        "PM10 vs PM2.5",
        "Robustness summary",
        "Main limitation"
    ],
    "interpretation": [
        "ENSO acts as a seasonal modulator rather than a single dominant driver with one uniform sign throughout the year.",
        "DJF remains the clearest benchmark season, but ENSO does not affect all metrics in the same way: onset frequency and synoptic organization tend to favor El Niño, whereas WHO exceedance frequency is higher under La Niña.",
        "MAM behaves as a true transition season. Although some descriptive ENSO differences appear in onset frequency and synoptic structure, the robustness tests do not support a strong phase-dependent exceedance or onset signal.",
        "JJA provides the clearest evidence of ENSO modulation. Both onset frequency and WHO exceedance frequency show robust phase dependence, and PM2.5 in particular displays a distinct ENSO-conditioned synoptic contrast.",
        "SON remains informative but secondary. For PM10, onset frequency and exceedance frequency show some ENSO dependence, while for PM2.5 the signal is more mixed and should be interpreted cautiously.",
        "ENSO modulation is expressed more clearly in exceedance frequency than in average exceedance magnitude, and more clearly in seasonal diagnostics than in pooled all-season summaries.",
        "PM2.5 shows a more seasonally extended ENSO sensitivity than PM10, especially in JJA and, to a lesser extent, SON. PM10 retains a clearer winter benchmark and a stronger Neutral preference from MAM to SON.",
        "The strongest and most defendible ENSO result in this stage is the JJA modulation signal. DJF is physically important but more internally split across onset, circulation, and exceedance metrics. MAM is transitional and less statistically robust.",
        "A small edge-effect limitation affects the ENSO assignment: 31 daily observations, mostly associated with DJF 2025 season-year logic, remained unlabeled because the ENSO lookup was truncated at 2024. These represent less than 1% of the full daily record and are unlikely to alter the main conclusions."
    ]
})

save_csv(enso_final_synthesis, "stage5_enso_final_synthesis_notes.csv", index=False)

print("Saved: stage5_enso_final_synthesis_notes.csv")
enso_final_synthesis

Saved: stage5_enso_final_synthesis_notes.csv


,theme,interpretation
0,Overall role of ENSO,ENSO acts as a seasonal modulator rather than ...
1,DJF,"DJF remains the clearest benchmark season, but..."
2,MAM,MAM behaves as a true transition season. Altho...
3,JJA,JJA provides the clearest evidence of ENSO mod...
4,SON,SON remains informative but secondary. For PM1...
5,Frequency vs burden,ENSO modulation is expressed more clearly in e...
6,PM10 vs PM2.5,PM2.5 shows a more seasonally extended ENSO se...
7,Robustness summary,The strongest and most defendible ENSO result ...
8,Main limitation,A small edge-effect limitation affects the ENS...


## 11. Final synthesis and export-ready products 

### 11.1 Summary tables

In [267]:
# ---------- A) Seasonal synthesis table ----------
# Keeping the most thesis-useful seasonal diagnostics in one place

# Day-0 summary (pivoted by pollutant)
day0_maxabs_pivot = seasonal_day0_comparison.pivot_table(
    index="season",
    columns="pollutant",
    values="max_abs_z500_anom"
).reset_index()
day0_maxabs_pivot.columns.name = None
day0_maxabs_pivot = day0_maxabs_pivot.rename(columns={
    "PM10": "PM10_day0_max_abs_z500",
    "PM2.5": "PM25_day0_max_abs_z500"
})

day0_meanabs_pivot = seasonal_day0_comparison.pivot_table(
    index="season",
    columns="pollutant",
    values="mean_abs_z500_anom"
).reset_index()
day0_meanabs_pivot.columns.name = None
day0_meanabs_pivot = day0_meanabs_pivot.rename(columns={
    "PM10": "PM10_day0_mean_abs_z500",
    "PM2.5": "PM25_day0_mean_abs_z500"
})

# Lag summary at lag0 and lag-1 already condensed in seasonal_lag_comparison
lag_summary_keep = seasonal_lag_comparison[
    [
        "pollutant", "season",
        "mean_abs_z500_lagm1",
        "mean_abs_z500_lag0",
        "mean_abs_z500_lagp1",
        "preconditioning_minus_onset",
        "persistence_minus_onset"
    ]
].copy()

lag0_pivot = lag_summary_keep.pivot_table(
    index="season",
    columns="pollutant",
    values="mean_abs_z500_lag0"
).reset_index()
lag0_pivot.columns.name = None
lag0_pivot = lag0_pivot.rename(columns={
    "PM10": "PM10_lag0_mean_abs_z500",
    "PM2.5": "PM25_lag0_mean_abs_z500"
})

lagm1_pivot = lag_summary_keep.pivot_table(
    index="season",
    columns="pollutant",
    values="mean_abs_z500_lagm1"
).reset_index()
lagm1_pivot.columns.name = None
lagm1_pivot = lagm1_pivot.rename(columns={
    "PM10": "PM10_lagm1_mean_abs_z500",
    "PM2.5": "PM25_lagm1_mean_abs_z500"
})

# Significance summary: keep lag 0 and lag -1 fractions by pollutant
signif_lag0 = seasonal_signif_summary.loc[seasonal_signif_summary["lag"] == 0].copy()
signif_lag0_pivot = signif_lag0.pivot_table(
    index="season",
    columns="pollutant",
    values="frac_significant_area"
).reset_index()
signif_lag0_pivot.columns.name = None
signif_lag0_pivot = signif_lag0_pivot.rename(columns={
    "PM10": "PM10_frac_signif_area_day0",
    "PM2.5": "PM25_frac_signif_area_day0"
})

signif_lagm1 = seasonal_signif_summary.loc[seasonal_signif_summary["lag"] == -1].copy()
signif_lagm1_pivot = signif_lagm1.pivot_table(
    index="season",
    columns="pollutant",
    values="frac_significant_area"
).reset_index()
signif_lagm1_pivot.columns.name = None
signif_lagm1_pivot = signif_lagm1_pivot.rename(columns={
    "PM10": "PM10_frac_signif_area_lagm1",
    "PM2.5": "PM25_frac_signif_area_lagm1"
})

# Main seasonal framework base
seasonal_thesis_summary = seasonal_anchor_table[
    [
        "season",
        "PM10_onset_days",
        "PM25_onset_days",
        "PM10_onset_pct_of_days",
        "PM25_onset_pct_of_days",
        "PM10_exceed_pct",
        "PM25_exceed_pct",
        "precip_vom_mean_mmday",
        "interpretation_anchor"
    ]
].copy()

# Merge everything
seasonal_thesis_summary = (
    seasonal_thesis_summary
    .merge(day0_maxabs_pivot, on="season", how="left")
    .merge(day0_meanabs_pivot, on="season", how="left")
    .merge(lag0_pivot, on="season", how="left")
    .merge(lagm1_pivot, on="season", how="left")
    .merge(signif_lag0_pivot, on="season", how="left")
    .merge(signif_lagm1_pivot, on="season", how="left")
)

save_csv(seasonal_thesis_summary, "stage5_final_seasonal_thesis_summary.csv", index=False)

print("Saved: stage5_final_seasonal_thesis_summary.csv")
seasonal_thesis_summary

Saved: stage5_final_seasonal_thesis_summary.csv


,season,PM10_onset_days,PM25_onset_days,PM10_onset_pct_of_days,PM25_onset_pct_of_days,PM10_exceed_pct,PM25_exceed_pct,precip_vom_mean_mmday,interpretation_anchor,PM10_day0_max_abs_z500,PM25_day0_max_abs_z500,PM10_day0_mean_abs_z500,PM25_day0_mean_abs_z500,PM10_lag0_mean_abs_z500,PM25_lag0_mean_abs_z500,PM10_lagm1_mean_abs_z500,PM25_lagm1_mean_abs_z500,PM10_frac_signif_area_day0,PM25_frac_signif_area_day0,PM10_frac_signif_area_lagm1,PM25_frac_signif_area_lagm1
0,DJF,70,74,6.216696,6.571936,73.855756,89.528705,0.354234,Dry benchmark season,14.240988,18.052292,4.260468,5.133082,4.260468,5.133082,3.355952,4.575150,0.576111,0.604921,0.481667,0.511190
1,MAM,53,63,4.592721,5.459272,66.547508,95.050711,1.163150,Transition season,32.756939,20.283266,4.054671,2.663425,4.054671,2.663425,6.494188,2.412752,0.542698,0.553889,0.518968,0.530397
2,JJA,68,76,5.892548,6.585789,14.789858,71.526699,4.688356,Wet-season contrast,9.755809,12.086612,2.886149,4.246170,2.886149,4.246170,1.548498,3.699394,0.625397,0.599206,0.704524,0.599206
3,SON,59,73,5.263158,6.512043,28.132614,71.006429,2.887631,Included for completeness,15.123734,22.961071,5.767316,6.560750,5.767316,6.560750,3.732033,4.770576,0.162302,0.304365,0.141667,0.187143


In [268]:
# ------------------------------------------------------------
# ENSO synthesis table
# ------------------------------------------------------------

# Onset-rate summary by ENSO
onset_rates_tidy = onset_counts_by_phase[
    [
        "pollutant", "season", "enso_phase",
        "n_onset_events", "n_season_years",
        "onset_events_per_season_year"
    ]
].copy()

# Onset robustness summary
onset_robustness_keep = onset_enso_robustness[
    ["pollutant", "season", "p_value", "significant_at_0p05", "result_note"]
].copy().rename(columns={
    "p_value": "onset_freq_p_value",
    "significant_at_0p05": "onset_freq_significant",
    "result_note": "onset_freq_note"
})

# WHO robustness summary
who_robustness_keep = who_enso_robustness[
    ["pollutant", "season", "p_value", "cramers_v", "significant_at_0p05", "result_note"]
].copy().rename(columns={
    "p_value": "who_freq_p_value",
    "cramers_v": "who_freq_cramers_v",
    "significant_at_0p05": "who_freq_significant",
    "result_note": "who_freq_note"
})

# WHO seasonal by ENSO phase -> split to long tidy by pollutant
pm10_who_enso = who_enso_seasonal[
    ["season", "enso_phase", "PM10_exceed_pct", "PM10_excess_mag_mean", "PM10_excess_load_sum"]
].copy()
pm10_who_enso["pollutant"] = "PM10"
pm10_who_enso = pm10_who_enso.rename(columns={
    "PM10_exceed_pct": "who_exceed_pct",
    "PM10_excess_mag_mean": "who_excess_mag_mean",
    "PM10_excess_load_sum": "who_excess_load_sum"
})

pm25_who_enso = who_enso_seasonal[
    ["season", "enso_phase", "PM25_exceed_pct", "PM25_excess_mag_mean", "PM25_excess_load_sum"]
].copy()
pm25_who_enso["pollutant"] = "PM2.5"
pm25_who_enso = pm25_who_enso.rename(columns={
    "PM25_exceed_pct": "who_exceed_pct",
    "PM25_excess_mag_mean": "who_excess_mag_mean",
    "PM25_excess_load_sum": "who_excess_load_sum"
})

who_enso_tidy = pd.concat([pm10_who_enso, pm25_who_enso], ignore_index=True)

# Merge to one final ENSO synthesis table
enso_thesis_summary = (
    onset_rates_tidy
    .merge(who_enso_tidy, on=["pollutant", "season", "enso_phase"], how="left")
    .merge(onset_robustness_keep, on=["pollutant", "season"], how="left")
    .merge(who_robustness_keep, on=["pollutant", "season"], how="left")
)

save_csv(enso_thesis_summary, "stage5_final_enso_thesis_summary.csv", index=False)

print("Saved: stage5_final_enso_thesis_summary.csv")
enso_thesis_summary.head(24)

Saved: stage5_final_enso_thesis_summary.csv


,pollutant,season,enso_phase,n_onset_events,n_season_years,onset_events_per_season_year,who_exceed_pct,who_excess_mag_mean,who_excess_load_sum,onset_freq_p_value,onset_freq_significant,onset_freq_note,who_freq_p_value,who_freq_cramers_v,who_freq_significant,who_freq_note
0,PM10,DJF,El Niño,20,3,6.666667,65.073529,20.414802,3613.42,0.462038,False,No strong evidence against phase-proportional ...,3.778905e-05,0.136382,True,Exceedance frequency differs by ENSO phase
1,PM10,DJF,Neutral,13,3,4.333333,72.693727,18.544365,3653.24,0.462038,False,No strong evidence against phase-proportional ...,3.778905e-05,0.136382,True,Exceedance frequency differs by ENSO phase
2,PM10,DJF,La Niña,37,7,5.285714,79.528986,19.030159,8354.24,0.462038,False,No strong evidence against phase-proportional ...,3.778905e-05,0.136382,True,Exceedance frequency differs by ENSO phase
3,PM10,MAM,El Niño,9,3,3.000000,65.942029,13.434231,2445.03,0.118378,False,No strong evidence against phase-proportional ...,1.901581e-01,0.053636,False,No strong evidence that exceedance frequency d...
4,PM10,MAM,Neutral,36,7,5.142857,67.701863,15.962271,6959.55,0.118378,False,No strong evidence against phase-proportional ...,1.901581e-01,0.053636,False,No strong evidence that exceedance frequency d...
5,PM10,MAM,La Niña,8,3,2.666667,61.111111,12.471119,1783.37,0.118378,False,No strong evidence against phase-proportional ...,1.901581e-01,0.053636,False,No strong evidence that exceedance frequency d...
6,PM10,JJA,El Niño,11,2,5.500000,14.673913,6.705185,181.04,0.031498,True,Phase-dependent onset frequency suggested,1.789160e-06,0.151445,True,Exceedance frequency differs by ENSO phase
7,PM10,JJA,Neutral,41,6,6.833333,19.021739,6.378381,669.73,0.031498,True,Phase-dependent onset frequency suggested,1.789160e-06,0.151445,True,Exceedance frequency differs by ENSO phase
8,PM10,JJA,La Niña,16,5,3.200000,7.416268,6.130323,190.04,0.031498,True,Phase-dependent onset frequency suggested,1.789160e-06,0.151445,True,Exceedance frequency differs by ENSO phase
9,PM10,SON,El Niño,18,3,6.000000,24.542125,13.165373,882.08,0.002298,True,Phase-dependent onset frequency suggested,5.390423e-03,0.096533,True,Exceedance frequency differs by ENSO phase


### 11.2 Figure bank and notebook closing note

In [269]:
figure_bank = pd.DataFrame({
    "filename": [
        "pm10_seasonal_day0_composites_500hPa.png",
        "pm25_seasonal_day0_composites_500hPa.png",
        "stage5_PM10_DJF_lag_composites_6panel.png",
        "stage5_PM10_MAM_lag_composites_6panel.png",
        "stage5_PM10_JJA_lag_composites_6panel.png",
        "stage5_PM25_DJF_lag_composites_6panel.png",
        "stage5_PM25_MAM_lag_composites_6panel.png",
        "stage5_PM25_JJA_lag_composites_6panel.png",
        "stage5_PM10_lagged_seasonal_diagnostics.png",
        "stage5_PM25_lagged_seasonal_diagnostics.png",
        "stage5_PM10_seasonal_significance_day0.png",
        "stage5_PM25_seasonal_significance_day0.png",
        "stage5_PM10_seasonal_significance_lagm1.png",
        "stage5_PM25_seasonal_significance_lagm1.png",
        "stage5_PM10_onset_rates_by_enso_phase.png",
        "stage5_PM25_onset_rates_by_enso_phase.png",
        "stage5_PM10_DJF_enso_phase_day0_3panel.png",
        "stage5_PM10_MAM_enso_phase_day0_3panel.png",
        "stage5_PM25_DJF_enso_phase_day0_3panel.png",
        "stage5_PM25_MAM_enso_phase_day0_3panel.png",
        "stage5_PM10_JJA_enso_phase_day0_3panel.png",
        "stage5_PM10_SON_enso_phase_day0_3panel.png",
        "stage5_PM25_JJA_enso_phase_day0_3panel.png",
        "stage5_PM25_SON_enso_phase_day0_3panel.png",
    ],
    "section": [
        "6", "6",
        "7", "7", "7",
        "7", "7", "7",
        "7.5", "7.5",
        "8", "8", "8", "8",
        "10.2", "10.2",
        "10.3", "10.3", "10.3", "10.3",
        "10.3", "10.3", "10.3", "10.3"
    ],
    "use_in": [
        "Core", "Core",
        "Core", "Core", "Appendix",
        "Core", "Core", "Core",
        "Core", "Core",
        "Core", "Core", "Appendix", "Appendix",
        "Core", "Core",
        "Core", "Core", "Core", "Core",
        "Appendix", "Appendix", "Appendix", "Appendix"
    ],
    "reason": [
        "Seasonal day-0 synoptic summary for PM10.",
        "Seasonal day-0 synoptic summary for PM2.5.",
        "Best winter benchmark sequence for PM10.",
        "Best transition-season sequence for PM10.",
        "Useful wet-season contrast, but secondary if slide space is limited.",
        "Best winter benchmark sequence for PM2.5.",
        "Best transition-season sequence for PM2.5.",
        "Useful wet-season contrast and PM10–PM2.5 divergence support.",
        "Compact seasonal mechanism support for PM10.",
        "Compact seasonal mechanism support for PM2.5.",
        "Day-0 significance support for PM10.",
        "Day-0 significance support for PM2.5.",
        "Precursor significance support for PM10; useful but secondary.",
        "Precursor significance support for PM2.5; useful but secondary.",
        "Clear ENSO-rate summary for PM10.",
        "Clear ENSO-rate summary for PM2.5.",
        "Primary ENSO synoptic benchmark season for PM10.",
        "Primary ENSO transition season for PM10.",
        "Primary ENSO synoptic benchmark season for PM2.5.",
        "Primary ENSO transition season for PM2.5.",
        "Descriptive ENSO support only; smaller sample support.",
        "Descriptive ENSO support only; smaller sample support.",
        "Descriptive ENSO support only; smaller sample support.",
        "Descriptive ENSO support only; smaller sample support.",
    ]
})

save_csv(figure_bank, "stage5_figure_bank_recommendations.csv", index=False)

notebook_closing_note = pd.DataFrame({
    "topic": [
        "Notebook role",
        "Main question answered",
        "Seasonal story",
        "ENSO story",
        "Deliberately excluded from core",
        "Important limitation",
        "Next step after notebook"
    ],
    "note": [
        "This notebook serves as the final thesis-stage analysis notebook.",
        "It tests whether seasonal onset-based synoptic stories are physically coherent and whether they are modulated by ENSO.",
        "DJF remains the clearest dry benchmark, MAM behaves as a true transition season, JJA provides the clearest wet-season contrast, and SON is retained as a secondary drying-season context.",
        "ENSO acts as a seasonal modulator rather than as a single dominant driver with one uniform sign; JJA provides the clearest robustness signal.",
        "850 hPa and SH850 mechanism diagnostics were deliberately left out of the final core analysis.",
        "Thirty-one daily observations at the DJF 2025 edge were left without ENSO assignment because the CPC RONI lookup was truncated at 2024.",
        "Write the thesis and prepare the final defense presentation."
    ]
})

save_csv(notebook_closing_note, "stage5_notebook_closing_note.csv", index=False)

print("Saved: stage5_figure_bank_recommendations.csv")
print("Saved: stage5_notebook_closing_note.csv")

display(figure_bank)
display(notebook_closing_note)

Saved: stage5_figure_bank_recommendations.csv
Saved: stage5_notebook_closing_note.csv


,filename,section,use_in,reason
0,pm10_seasonal_day0_composites_500hPa.png,6,Core,Seasonal day-0 synoptic summary for PM10.
1,pm25_seasonal_day0_composites_500hPa.png,6,Core,Seasonal day-0 synoptic summary for PM2.5.
2,stage5_PM10_DJF_lag_composites_6panel.png,7,Core,Best winter benchmark sequence for PM10.
3,stage5_PM10_MAM_lag_composites_6panel.png,7,Core,Best transition-season sequence for PM10.
4,stage5_PM10_JJA_lag_composites_6panel.png,7,Appendix,"Useful wet-season contrast, but secondary if s..."
5,stage5_PM25_DJF_lag_composites_6panel.png,7,Core,Best winter benchmark sequence for PM2.5.
6,stage5_PM25_MAM_lag_composites_6panel.png,7,Core,Best transition-season sequence for PM2.5.
7,stage5_PM25_JJA_lag_composites_6panel.png,7,Core,Useful wet-season contrast and PM10–PM2.5 dive...
8,stage5_PM10_lagged_seasonal_diagnostics.png,7.5,Core,Compact seasonal mechanism support for PM10.
9,stage5_PM25_lagged_seasonal_diagnostics.png,7.5,Core,Compact seasonal mechanism support for PM2.5.


,topic,note
0,Notebook role,This notebook serves as the final thesis-stage...
1,Main question answered,It tests whether seasonal onset-based synoptic...
2,Seasonal story,"DJF remains the clearest dry benchmark, MAM be..."
3,ENSO story,ENSO acts as a seasonal modulator rather than ...
4,Deliberately excluded from core,850 hPa and SH850 mechanism diagnostics were d...
5,Important limitation,Thirty-one daily observations at the DJF 2025 ...
6,Next step after notebook,Write the thesis and prepare the final defense...
